# DAETF-Net - Domain-Adaptive Equivariant Tensor Fusion Network

Hyperspectral (HSI) and multispectral (MSI) image fusion built to survive the
transfer from the dataset it was trained on to one it has never seen.

**The problem this attacks.** Re-running ten published fusion methods under one
protocol shows spatial quality holds up across datasets while *spectral* quality
collapses: SAM is 2-7 deg in-domain and 8-36 deg cross-domain, and ERGAS rises by
one to two orders of magnitude. PSNR hides this - several baselines score
*higher* PSNR on the harder dataset - which is why the headline comparison in the
earlier benchmark was misleading.

**What is new here**

| Component | Mechanism | Verified by |
|---|---|---|
| EFE | p4 group-equivariant convolutions (lifting + group conv + group pool) | `check_equivariance` ~1e-6 |
| TSSE | Tucker contraction against a learned core tensor | `check_core_used` (gradient reaches the core) |
| AF-MoE | per-pixel top-k expert routing + load balancing | usage histogram |
| FDRM | orthonormal Haar DWT, learnable per-subband shrinkage, exact IDWT | `check_wavelet` ~1e-7 |
| DAE | degradation code conditioning every block through FiLM | auxiliary regression head |
| BPU | back-projection upsampler replacing bicubic | ablation |
| SPC loss | Charbonnier + SAM + gradient + SSIM + two physics terms | ablation |

**The central idea.** The two physics terms, `||Down(Y) - LR||` and
`||SRF(Y) - MSI||`, need no ground truth. They can be evaluated on any scene
from any sensor, so the same objective that trains the model also *adapts* it at
test time on a dataset with no labels.

**Hardware.** Fits a single 16 GB GPU. Choose **GPU T4 x2** as the accelerator:
PyTorch >= 2.6 ships no kernels for the P100's `sm_60`, so a P100 raises
"no kernel image is available" on the first CUDA op regardless of the code. The
environment cell below checks this and tells you if the accelerator is wrong.

Set `QUICK` below to smoke-test the whole pipeline in a few minutes before
committing to a full run.

## 1. Environment

In [ ]:
import os, sys, json, time, math, warnings
warnings.filterwarnings('ignore', category=UserWarning)

import torch, numpy as np
print('python  ', sys.version.split()[0])
print('torch   ', torch.__version__)
print('numpy   ', np.__version__)
print('cuda    ', torch.cuda.is_available())

GPU_OK = False
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    arch = f'sm_{p.major}{p.minor}'
    built = list(torch.cuda.get_arch_list())
    print('gpu     ', p.name, f'{p.total_memory / 2**30:.1f} GB', arch)
    print('built   ', ' '.join(built))
    GPU_OK = arch in built
    if not GPU_OK:
        # PyTorch >= 2.6 dropped Pascal (sm_60/sm_61), so a P100 cannot run
        # the current Kaggle image no matter what the code does. Catch it here
        # rather than 50 seconds later inside a forward pass.
        print()
        print('*' * 74)
        print(f'INCOMPATIBLE GPU: this torch build ships no kernels for {arch}.')
        print(f'  {p.name} is {arch}; this build targets: {" ".join(built)}')
        print('  Any CUDA op will raise "no kernel image is available".')
        print()
        print('  FIX: Notebook menu -> Settings -> Accelerator -> "GPU T4 x2"')
        print('       (T4 is sm_75 and is supported). Then re-run all.')
        print('*' * 74)
    else:
        # fp16 is worth enabling on any supported card here; tensor cores
        # (sm_70+) make it faster still, Pascal only saves memory.
        print('amp     ', 'fp16 with tensor cores' if p.major >= 7
              else 'fp16 (memory only, no tensor cores)')
else:
    print('no GPU detected - training will be extremely slow on CPU')

WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
os.chdir(WORK)
print('workdir ', os.getcwd())

## 2. The DAETF-Net package

Written out module by module so the notebook is fully self-contained - no
`git clone`, no internet needed. Generated from `proposal1/daetf/` by
`tools/build_notebook.py`; edit the package and regenerate rather than editing
these cells.

In [ ]:
import os; os.makedirs('daetf', exist_ok=True)

In [ ]:
%%writefile daetf/io_utils.py
"""Filesystem discovery and .mat reading.

This module deliberately has no dependency on the rest of the package so that
dataset discovery can never create an import cycle.
"""

from __future__ import annotations

import glob
import os
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np

try:
    import scipy.io as sio
except ImportError:  # pragma: no cover
    sio = None

SPLIT_NAMES = ("Train", "train", "TRAIN")
TEST_NAMES = ("Test", "test", "TEST", "Val", "val")


# --------------------------------------------------------------------------- IO
def load_mat(path: str) -> np.ndarray:
    """Return the first real array stored in a MATLAB file."""
    if sio is None:
        raise RuntimeError("scipy is required to read .mat files")
    mat = sio.loadmat(path)
    for k, v in mat.items():
        if not k.startswith("__") and isinstance(v, np.ndarray) and v.ndim >= 2:
            return np.asarray(v)
    raise ValueError(f"no array in {path}")


def to_chw01(arr: np.ndarray, channels: int) -> np.ndarray:
    """Normalise an array to channel-first float32 in [0, 1]."""
    if channels is None:
        raise ValueError("channel count is unresolved - call Config.resolve() first")
    a = np.squeeze(np.asarray(arr)).astype(np.float32)
    if a.ndim != 3:
        raise ValueError(f"expected 3D array, got {a.shape}")
    if a.shape[0] == channels:
        pass
    elif a.shape[-1] == channels:
        a = np.transpose(a, (2, 0, 1))
    else:
        raise ValueError(f"cannot find {channels} channels in {a.shape}")
    mx = float(a.max())
    if mx > 1.0:
        a = a / mx
    return np.clip(a, 0.0, 1.0)


# ------------------------------------------------------------------- discovery
def _looks_like_dataset(path: str) -> bool:
    """A dataset root is any directory holding <split>/HSI."""
    for split in SPLIT_NAMES + TEST_NAMES:
        d = os.path.join(path, split)
        if os.path.isdir(d) and any(
            os.path.isdir(os.path.join(d, h)) for h in ("HSI", "hsi")
        ):
            return True
    return False


def search_roots() -> List[str]:
    """Base locations to search under, most specific first.

    DAETF_DATA_ROOTS (os.pathsep separated) always takes priority, so discovery
    can be overridden without editing any code.
    """
    roots: List[str] = []
    env = os.environ.get("DAETF_DATA_ROOTS", "")
    roots += [p for p in env.split(os.pathsep) if p]
    roots += ["/kaggle/input"]
    roots += [os.path.join(os.getcwd(), "data"), os.getcwd()]
    return [r for r in roots if os.path.isdir(r)]


def find_dataset_roots(base: str, max_depth: int = 5) -> List[str]:
    """Breadth-first search under `base` for directories exposing <split>/HSI.

    Kaggle does not mount datasets at a predictable depth: attaching
    `owner/cave-dataset-2` can appear as /kaggle/input/cave-dataset-2/Data or
    as /kaggle/input/datasets/owner/cave-dataset-2/Data depending on how the
    kernel was configured. Searching a fixed depth silently fails on the
    second layout, so walk until a dataset is found.

    Only directories are visited, HSI/RGB leaves are never descended into, and
    the search stops descending as soon as a root matches - so this stays cheap
    even when the tree holds thousands of .mat files.
    """
    found: List[str] = []
    queue: List[Tuple[str, int]] = [(base, 0)]
    seen = set()
    while queue:
        path, depth = queue.pop(0)
        real = os.path.realpath(path)
        if real in seen:
            continue
        seen.add(real)
        if _looks_like_dataset(path):
            found.append(path)
            continue                      # do not descend into a match
        if depth >= max_depth:
            continue
        try:
            for entry in sorted(os.scandir(path), key=lambda e: e.name):
                if entry.is_dir(follow_symlinks=False) and \
                        entry.name not in ("HSI", "hsi", "RGB", "rgb"):
                    queue.append((entry.path, depth + 1))
        except OSError:
            continue
    return found


def discover_dataset(hints: Sequence[str] = (), required: bool = True,
                     verbose: bool = True) -> Optional[str]:
    """Locate a dataset root whose path matches one of `hints`.

    Handles `<root>/Data/Train/HSI`, `<root>/Train/HSI` and arbitrarily nested
    Kaggle mount points.
    """
    found: List[str] = []
    for root in search_roots():
        for cand in find_dataset_roots(root):
            if cand not in found:
                found.append(cand)
    if hints:
        lowered = [h.lower() for h in hints]
        ranked = [f for f in found if any(h in f.lower() for h in lowered)]
        found = ranked or found
    if not found:
        if required:
            listing = []
            for r in search_roots():
                try:
                    listing.append(f"{r} -> {sorted(os.listdir(r))[:8]}")
                except OSError:
                    pass
            raise FileNotFoundError(
                f"no dataset matching {list(hints)} found.\n"
                f"Searched (depth 5) under:\n  " + "\n  ".join(listing) +
                "\nA dataset root must contain <split>/HSI, e.g. Data/Train/HSI.\n"
                "Set DAETF_DATA_ROOTS or pass Config(source_root=...) explicitly."
            )
        if verbose:
            print(f"[config] optional dataset {list(hints)} not found - skipping")
        return None
    if verbose:
        print(f"[config] using dataset root: {found[0]}")
    return found[0]


def available_splits(root: str) -> Dict[str, str]:
    """Map canonical split name -> the directory name actually present."""
    out = {}
    for canonical, names in (("Train", SPLIT_NAMES), ("Test", TEST_NAMES)):
        for n in names:
            if os.path.isdir(os.path.join(root, n)):
                out[canonical] = n
                break
    return out


def infer_channels(root: str) -> Tuple[int, int]:
    """Read one HSI/RGB pair and report their channel counts."""
    splits = available_splits(root)
    split = splits.get("Train") or splits.get("Test")
    if split is None:
        raise FileNotFoundError(f"no usable split under {root}")
    base = os.path.join(root, split)
    hsi_dir = next(os.path.join(base, d) for d in ("HSI", "hsi")
                   if os.path.isdir(os.path.join(base, d)))
    rgb_dir = next((os.path.join(base, d) for d in ("RGB", "rgb")
                    if os.path.isdir(os.path.join(base, d))), None)
    hsi = np.squeeze(load_mat(sorted(glob.glob(os.path.join(hsi_dir, "*.mat")))[0]))
    bands = int(min(hsi.shape))
    msi_bands = 3
    if rgb_dir:
        rgb = np.squeeze(load_mat(sorted(glob.glob(os.path.join(rgb_dir, "*.mat")))[0]))
        msi_bands = int(min(rgb.shape))
    return bands, msi_bands


def find_pairs(root: str, split: str) -> List[Tuple[str, str, str]]:
    """Matched (stem, hsi_path, rgb_path) triples for a canonical split name."""
    actual = available_splits(root).get(split, split)
    base = os.path.join(root, actual)
    hsi_dir = next((os.path.join(base, d) for d in ("HSI", "hsi")
                    if os.path.isdir(os.path.join(base, d))), None)
    rgb_dir = next((os.path.join(base, d) for d in ("RGB", "rgb")
                    if os.path.isdir(os.path.join(base, d))), None)
    if not hsi_dir or not rgb_dir:
        raise FileNotFoundError(f"no HSI/RGB folders under {base}")
    rgb = {os.path.splitext(os.path.basename(p))[0]: p
           for p in glob.glob(os.path.join(rgb_dir, "*.mat"))}
    out = []
    for h in sorted(glob.glob(os.path.join(hsi_dir, "*.mat"))):
        stem = os.path.splitext(os.path.basename(h))[0]
        if stem in rgb:
            out.append((stem, h, rgb[stem]))
    if not out:
        raise RuntimeError(f"no matched pairs under {base}")
    return out

In [ ]:
%%writefile daetf/config.py
"""Experiment configuration.

Every path and channel count defaults to None and is resolved by inspecting the
filesystem, so nothing about a particular machine or Kaggle dataset slug is
baked into the code.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass
from typing import Optional, Sequence, Tuple

from .io_utils import discover_dataset, infer_channels


@dataclass
class Config:
    # --- data (None => auto-discover) --------------------------------------
    source_root: Optional[str] = None    # domain the model trains on
    target_root: Optional[str] = None    # unseen domain used for transfer tests
    bands: Optional[int] = None
    msi_bands: Optional[int] = None
    scale: int = 4                       # super-resolution factor
    patch: int = 64                      # HR training patch (multiple of scale)

    # --- model --------------------------------------------------------------
    width: int = 64                      # main feature width
    equi_width: int = 16                 # per-orientation width in the p4 stem
    equi_depth: int = 2                  # number of p4 -> p4 group convolutions
    rank: int = 16                       # Tucker ranks (R1 = R2 = R3)
    experts: int = 4
    topk: int = 2
    bp_iters: int = 2                    # back-projection refinement steps
    code_dim: int = 128                  # degradation code width
    blur_ksize: int = 9                  # support of the simulated blur kernels

    # --- degradation simulation (domain randomisation) ----------------------
    sigma_range: Tuple[float, float] = (0.6, 2.4)
    aniso: float = 0.5                   # probability of an anisotropic kernel
    noise_range: Tuple[float, float] = (0.0, 0.03)
    srf_jitter: float = 0.35             # probability of a jittered synthetic MSI
    eval_sigma: float = 1.2              # fixed kernel used to build the eval LR

    # --- optimisation --------------------------------------------------------
    iters: int = 20000
    batch: int = 16
    lr: float = 2e-4
    min_lr: float = 1e-6
    warmup: int = 500
    grad_clip: float = 1.0
    amp: bool = True                     # fp16: halves memory, faster on sm_70+
    workers: int = 2
    seed: int = 42

    # --- loss weights --------------------------------------------------------
    w_char: float = 1.0
    w_sam: float = 0.30
    w_grad: float = 0.20
    w_ssim: float = 0.15
    w_spat: float = 0.50                 # || Down(Y) - LR ||
    w_spec: float = 0.50                 # || SRF(Y)  - MSI ||
    w_bal: float = 0.01
    w_rank: float = 1e-4
    w_deg: float = 0.05
    w_mmd: float = 0.10

    # --- ablation switches (all modules on by default) ----------------------
    use_equivariant: bool = True
    use_tsse: bool = True
    use_moe: bool = True
    use_fdrm: bool = True
    use_backprojection: bool = True
    use_physics: bool = True
    use_degradation_code: bool = True

    # --- bookkeeping ---------------------------------------------------------
    out_dir: str = "./daetf_out"
    val_every: int = 1000
    log_every: int = 100
    val_scenes: int = 4

    def __post_init__(self) -> None:
        assert self.patch % self.scale == 0, "patch must be divisible by scale"
        assert self.topk <= self.experts, "topk cannot exceed the number of experts"

    def resolve(self, source_hints: Sequence[str] = ("cave",),
                target_hints: Sequence[str] = ("harvard",),
                verbose: bool = True) -> "Config":
        """Fill in any field still set to None. Idempotent."""
        if self.source_root is None:
            self.source_root = discover_dataset(source_hints, verbose=verbose)
        if self.target_root is None:
            self.target_root = discover_dataset(target_hints, required=False,
                                                verbose=verbose)
        if self.bands is None or self.msi_bands is None:
            b, m = infer_channels(self.source_root)
            self.bands = self.bands or b
            self.msi_bands = self.msi_bands or m
            if verbose:
                print(f"[config] inferred bands={self.bands} msi_bands={self.msi_bands}")
        return self

    def to_dict(self) -> dict:
        return asdict(self)

In [ ]:
%%writefile daetf/degrade.py
"""Forward observation model: blur, decimation and the fixed evaluation operator.

Keeping the degradation differentiable is what allows the spatial-consistency
term of the loss to be back-propagated through, and therefore what allows
self-supervised adaptation on a domain with no ground truth.
"""

from __future__ import annotations

import math
from typing import TYPE_CHECKING, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

if TYPE_CHECKING:  # pragma: no cover
    from .config import Config


def gaussian_kernel2d(ksize: int, sx: float, sy: float, theta: float) -> torch.Tensor:
    """Rotated anisotropic Gaussian blur kernel, normalised to sum 1."""
    ax = torch.arange(ksize, dtype=torch.float32) - (ksize - 1) / 2.0
    yy, xx = torch.meshgrid(ax, ax, indexing="ij")
    cos_t, sin_t = math.cos(theta), math.sin(theta)
    xr = xx * cos_t + yy * sin_t
    yr = -xx * sin_t + yy * cos_t
    k = torch.exp(-0.5 * ((xr / sx) ** 2 + (yr / sy) ** 2))
    return k / k.sum().clamp_min(1e-12)


def blur_downsample(x: torch.Tensor, kernel: torch.Tensor, scale: int) -> torch.Tensor:
    """Apply a per-sample blur kernel then decimate. Differentiable.

    x      : [B, C, H, W]
    kernel : [k, k] (shared) or [B, k, k] (one kernel per sample)
    """
    b, c, _, _ = x.shape
    if kernel.dim() == 2:
        kernel = kernel.unsqueeze(0).expand(b, -1, -1)
    k = kernel.shape[-1]
    pad = k // 2
    # fold the batch into the channel axis so each sample keeps its own kernel
    w = kernel.to(x.dtype).reshape(b, 1, 1, k, k).expand(b, c, 1, k, k).reshape(b * c, 1, k, k)
    xr = x.reshape(1, b * c, *x.shape[-2:])
    xr = F.pad(xr, (pad, pad, pad, pad), mode="reflect")
    out = F.conv2d(xr, w, groups=b * c)
    out = out.reshape(b, c, *out.shape[-2:])
    return out[..., ::scale, ::scale].contiguous()


class FixedDegradation(nn.Module):
    """Non-learnable blur+decimate: builds the evaluation LR input and backs the
    spatial-consistency loss."""

    def __init__(self, scale: int, ksize: int = 9, sigma: float = 1.2):
        super().__init__()
        self.scale = scale
        self.register_buffer("kernel", gaussian_kernel2d(ksize, sigma, sigma, 0.0))

    @classmethod
    def from_config(cls, cfg: "Config") -> "FixedDegradation":
        return cls(cfg.scale, ksize=cfg.blur_ksize, sigma=cfg.eval_sigma)

    def forward(self, x: torch.Tensor, kernel: Optional[torch.Tensor] = None
                ) -> torch.Tensor:
        k = self.kernel if kernel is None else kernel
        return blur_downsample(x, k, self.scale)

In [ ]:
%%writefile daetf/metrics.py
"""One metric implementation, shared by every method and both datasets.

The v1 benchmark computed PSNR/SSIM/SAM/ERGAS separately inside each of the 20
notebooks, with different data ranges, different normalisations and ERGAS scale
factors that did not always match the actual downsampling factor. Those numbers
were therefore not comparable across methods. Everything here is fixed:

  * PSNR uses a constant data_range (default 1.0), never the per-image maximum,
    which otherwise inflates scores on dark scenes.
  * SSIM is Gaussian-windowed (11x11, sigma 1.5) and averaged over bands.
  * SAM is reported in degrees, ignoring degenerate zero-spectra pixels.
  * ERGAS receives the true scale factor of the experiment.
"""

from __future__ import annotations

from typing import Dict

import numpy as np
import torch
import torch.nn.functional as F


def _gauss_window(size: int, sigma: float, device, dtype) -> torch.Tensor:
    coords = torch.arange(size, device=device, dtype=dtype) - size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    return g[:, None] @ g[None, :]


def ssim_torch(pred: torch.Tensor, target: torch.Tensor, data_range: float = 1.0,
               size: int = 11, sigma: float = 1.5) -> torch.Tensor:
    """Gaussian-windowed SSIM, averaged over channels. Differentiable."""
    c = pred.shape[1]
    win = _gauss_window(size, sigma, pred.device, pred.dtype).expand(c, 1, size, size)
    mu1 = F.conv2d(pred, win, padding=size // 2, groups=c)
    mu2 = F.conv2d(target, win, padding=size // 2, groups=c)
    mu1s, mu2s, mu12 = mu1 ** 2, mu2 ** 2, mu1 * mu2
    s1 = F.conv2d(pred * pred, win, padding=size // 2, groups=c) - mu1s
    s2 = F.conv2d(target * target, win, padding=size // 2, groups=c) - mu2s
    s12 = F.conv2d(pred * target, win, padding=size // 2, groups=c) - mu12
    c1, c2 = (0.01 * data_range) ** 2, (0.03 * data_range) ** 2
    m = ((2 * mu12 + c1) * (2 * s12 + c2)) / ((mu1s + mu2s + c1) * (s1 + s2 + c2))
    return m.mean()


def _hwc(x: np.ndarray) -> np.ndarray:
    return x if x.shape[-1] <= 64 else np.transpose(x, (1, 2, 0))


def metric_psnr(pred: np.ndarray, ref: np.ndarray, data_range: float = 1.0) -> float:
    mse = float(np.mean((pred - ref) ** 2))
    return 99.0 if mse <= 1e-12 else float(10 * np.log10(data_range ** 2 / mse))


def metric_sam(pred: np.ndarray, ref: np.ndarray, eps: float = 1e-8) -> float:
    p, r = _hwc(pred).reshape(-1, pred.shape[-1]), _hwc(ref).reshape(-1, ref.shape[-1])
    cos = (p * r).sum(1) / np.maximum(np.linalg.norm(p, axis=1) * np.linalg.norm(r, axis=1), eps)
    ang = np.degrees(np.arccos(np.clip(cos, -1, 1)))
    return float(np.mean(ang[np.isfinite(ang)]))


def metric_ergas(pred: np.ndarray, ref: np.ndarray, scale: int, eps: float = 1e-8) -> float:
    p, r = _hwc(pred), _hwc(ref)
    rmse = np.sqrt(np.mean((p - r) ** 2, axis=(0, 1)))
    mu = np.maximum(np.mean(r, axis=(0, 1)), eps)
    return float(100.0 / scale * np.sqrt(np.mean((rmse / mu) ** 2)))


def metric_ssim(pred: np.ndarray, ref: np.ndarray, data_range: float = 1.0) -> float:
    p = torch.from_numpy(np.ascontiguousarray(_hwc(pred).transpose(2, 0, 1)))[None].float()
    r = torch.from_numpy(np.ascontiguousarray(_hwc(ref).transpose(2, 0, 1)))[None].float()
    return float(ssim_torch(p, r, data_range=data_range))


def evaluate_arrays(pred: np.ndarray, ref: np.ndarray, scale: int) -> Dict[str, float]:
    """The four reported metrics for one scene."""
    return {
        "psnr": metric_psnr(pred, ref),
        "ssim": metric_ssim(pred, ref),
        "sam": metric_sam(pred, ref),
        "ergas": metric_ergas(pred, ref, scale),
    }

In [ ]:
%%writefile daetf/modules.py
"""Network building blocks.

Each block below implements the mechanism the design document claims, and the
claim is checked numerically in `selfcheck.py` rather than asserted in prose:

  P4ConvZ2 / P4ConvP4          -> genuine p4 equivariance   (checked to ~1e-6)
  HaarDWT                      -> exact orthonormal inverse (checked to ~1e-7)
  TensorSpectralSpatialEncoder -> the Tucker core carries gradient (checked)
"""

from __future__ import annotations

from typing import Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F


# --------------------------------------------------------------------------- EFE
class P4ConvZ2(nn.Module):
    """Lifting convolution Z2 -> p4. The output carries an explicit orientation
    axis of size 4, produced by convolving with the four rotated copies of one
    shared kernel."""

    def __init__(self, in_ch: int, out_ch: int, ksize: int = 3, bias: bool = True):
        super().__init__()
        self.out_ch, self.ksize = out_ch, ksize
        self.weight = nn.Parameter(torch.empty(out_ch, in_ch, ksize, ksize))
        nn.init.kaiming_normal_(self.weight, mode="fan_out", nonlinearity="relu")
        self.bias = nn.Parameter(torch.zeros(out_ch)) if bias else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:  # [B,Cin,H,W]->[B,Cout,4,H,W]
        w = torch.cat([torch.rot90(self.weight, r, dims=(2, 3)) for r in range(4)], dim=0)
        b = None if self.bias is None else self.bias.repeat(4)
        y = F.conv2d(x, w, b, padding=self.ksize // 2)
        bsz, _, h, wd = y.shape
        return y.view(bsz, 4, self.out_ch, h, wd).transpose(1, 2)


class P4ConvP4(nn.Module):
    """Group convolution p4 -> p4.

    For output orientation r the filter is rotated in space *and* cyclically
    shifted along the orientation axis; doing only one of the two is the usual
    way a 'rotation-equivariant' layer silently fails to be equivariant.
    """

    def __init__(self, in_ch: int, out_ch: int, ksize: int = 3, bias: bool = True):
        super().__init__()
        self.in_ch, self.out_ch, self.ksize = in_ch, out_ch, ksize
        self.weight = nn.Parameter(torch.empty(out_ch, in_ch, 4, ksize, ksize))
        nn.init.kaiming_normal_(self.weight.view(out_ch, -1, ksize, ksize),
                                mode="fan_out", nonlinearity="relu")
        self.bias = nn.Parameter(torch.zeros(out_ch)) if bias else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:  # [B,Cin,4,H,W]->[B,Cout,4,H,W]
        bsz, cin, _, h, wd = x.shape
        xf = x.reshape(bsz, cin * 4, h, wd)
        ws = []
        for r in range(4):
            wr = torch.rot90(self.weight, r, dims=(3, 4))   # rotate the spatial support
            wr = torch.roll(wr, shifts=r, dims=2)           # act on the orientation axis
            ws.append(wr.reshape(self.out_ch, cin * 4, self.ksize, self.ksize))
        w = torch.cat(ws, dim=0)
        b = None if self.bias is None else self.bias.repeat(4)
        y = F.conv2d(xf, w, b, padding=self.ksize // 2)
        return y.view(bsz, 4, self.out_ch, h, wd).transpose(1, 2)


class EquivariantFeatureExtractor(nn.Module):
    """EFE. BatchNorm3d shares statistics across orientations, so normalisation
    does not break equivariance; the closing max over the orientation axis makes
    the output a plain feature map that rotates with the input."""

    def __init__(self, in_ch: int, width: int, out_ch: int, depth: int = 2):
        super().__init__()
        self.lift = P4ConvZ2(in_ch, width)
        self.bn0 = nn.BatchNorm3d(width)
        self.blocks = nn.ModuleList([
            nn.ModuleList([P4ConvP4(width, width), nn.BatchNorm3d(width)])
            for _ in range(depth)
        ])
        self.proj = nn.Conv2d(width, out_ch, 1)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.act(self.bn0(self.lift(x)))
        for conv, bn in self.blocks:
            h = self.act(bn(conv(h))) + h
        h = h.max(dim=2).values          # group pooling over the four orientations
        return self.proj(h)


class PlainFeatureExtractor(nn.Module):
    """Non-equivariant control arm for the EFE ablation: matched depth and
    parameter budget, ordinary convolutions."""

    def __init__(self, in_ch: int, width: int, out_ch: int, depth: int = 2):
        super().__init__()
        layers = [nn.Conv2d(in_ch, width * 4, 3, 1, 1), nn.ReLU(inplace=True)]
        for _ in range(depth):
            layers += [nn.Conv2d(width * 4, width * 4, 3, 1, 1), nn.ReLU(inplace=True)]
        layers += [nn.Conv2d(width * 4, out_ch, 1)]
        self.body = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.body(x)


# --------------------------------------------------------------- degradation code
class DegradationEncoder(nn.Module):
    """Estimates a degradation code from the observed (LR-HSI, MSI) pair.

    An auxiliary head regresses the true degradation parameters, which are known
    during training because we synthesise them. Without that supervision the
    code tends to collapse to a constant and the conditioning does nothing.
    """

    def __init__(self, hsi_ch: int, msi_ch: int, code: int = 128):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(hsi_ch + msi_ch, 64, 3, 2, 1), nn.LeakyReLU(0.1, True),
            nn.Conv2d(64, 96, 3, 2, 1), nn.LeakyReLU(0.1, True),
            nn.Conv2d(96, 128, 3, 1, 1), nn.LeakyReLU(0.1, True),
        )
        self.head = nn.Sequential(nn.Linear(256, code), nn.LeakyReLU(0.1, True),
                                  nn.Linear(code, code))
        self.deg_head = nn.Linear(code, 5)   # sx, sy, sin2t, cos2t, noise

    def forward(self, lr_hsi: torch.Tensor, msi: torch.Tensor
                ) -> Tuple[torch.Tensor, torch.Tensor]:
        msi_lr = F.adaptive_avg_pool2d(msi, lr_hsi.shape[-2:])
        f = self.body(torch.cat([lr_hsi, msi_lr], dim=1))
        stats = torch.cat([f.mean(dim=(2, 3)), f.amax(dim=(2, 3))], dim=1)
        code = self.head(stats)
        return code, self.deg_head(code)


class FiLM(nn.Module):
    """Feature-wise linear modulation. Zero-initialised so the conditioned model
    starts exactly at the unconditioned one."""

    def __init__(self, code: int, channels: int):
        super().__init__()
        self.fc = nn.Linear(code, channels * 2)
        nn.init.zeros_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x: torch.Tensor, code: torch.Tensor) -> torch.Tensor:
        gamma, beta = self.fc(code).chunk(2, dim=1)
        return x * (1 + gamma[:, :, None, None]) + beta[:, :, None, None]


# -------------------------------------------------------------------------- TSSE
class TensorSpectralSpatialEncoder(nn.Module):
    """z[b,k,h,w] = sum_{i,j} G[i,j,k] * a[b,i,h,w] * b[b,j,h,w]

    A Tucker-style contraction of the outer product of the two projected feature
    maps against a learned core tensor G, realised as a 1x1 convolution whose
    weights *are* G - so the core genuinely participates and receives gradient.
    """

    def __init__(self, hsi_ch: int, msi_ch: int, out_ch: int, rank: int = 16):
        super().__init__()
        self.rank = rank
        self.proj_hsi = nn.Conv2d(hsi_ch, rank, 1)
        self.proj_msi = nn.Conv2d(msi_ch, rank, 1)
        self.core = nn.Parameter(torch.randn(rank, rank, rank) * (rank ** -0.75))
        self.out = nn.Sequential(nn.Conv2d(rank, out_ch, 1), nn.LeakyReLU(0.1, True),
                                 nn.Conv2d(out_ch, out_ch, 3, 1, 1))
        self.skip = nn.Conv2d(hsi_ch + msi_ch, out_ch, 1)
        self.norm = nn.GroupNorm(8, out_ch)

    def forward(self, a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
        pa = self.proj_hsi(a)                        # [B,R1,H,W]
        pb = self.proj_msi(b)                        # [B,R2,H,W]
        outer = (pa.unsqueeze(2) * pb.unsqueeze(1)).flatten(1, 2)   # [B,R1*R2,H,W]
        core = self.core.permute(2, 0, 1).reshape(self.rank, self.rank * self.rank, 1, 1)
        z = F.conv2d(outer, core.to(outer.dtype))    # [B,R3,H,W]
        return self.norm(self.out(z) + self.skip(torch.cat([a, b], dim=1)))

    def rank_penalty(self) -> torch.Tensor:
        """Nuclear norm of the mode-3 unfolding: a convex surrogate for rank."""
        return torch.linalg.svdvals(self.core.reshape(self.rank, -1).float()).sum()


# ------------------------------------------------------------------------- AF-MoE
class RegionAwareMoE(nn.Module):
    """Per-pixel top-k expert routing.

    A globally pooled gate must commit to one fusion strategy for a whole image.
    Routing per pixel lets shadowed, textured and flat regions of the same scene
    take different experts, which is the property the region-aware MoE
    literature reports as the win.
    """

    def __init__(self, channels: int, experts: int = 4, topk: int = 2):
        super().__init__()
        self.experts_n, self.topk = experts, min(topk, experts)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(channels, channels, 3, 1, 1), nn.LeakyReLU(0.1, True),
                nn.Conv2d(channels, channels, 3, 1, 1),
            ) for _ in range(experts)
        ])
        self.gate = nn.Sequential(
            nn.Conv2d(channels, channels // 2, 3, 1, 1), nn.LeakyReLU(0.1, True),
            nn.Conv2d(channels // 2, experts, 1),
        )
        self.last_gate: Optional[torch.Tensor] = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        logits = self.gate(x)                              # [B,E,H,W]
        if self.topk < self.experts_n:
            thresh = logits.topk(self.topk, dim=1).values[:, -1:, :, :]
            logits = logits.masked_fill(logits < thresh, float("-inf"))
        g = logits.softmax(dim=1)
        self.last_gate = g
        out = x
        for i, expert in enumerate(self.experts):
            out = out + g[:, i: i + 1] * expert(x)
        return out

    def balance_loss(self) -> torch.Tensor:
        """Squared coefficient of variation of expert usage; 0 when uniform.
        Without it, top-k routing collapses onto a single expert."""
        if self.last_gate is None:
            return torch.zeros((), device=next(self.parameters()).device)
        imp = self.last_gate.mean(dim=(0, 2, 3))
        return self.experts_n * (imp ** 2).sum() - 1.0

    @torch.no_grad()
    def usage(self) -> Optional[torch.Tensor]:
        """Per-expert mean gate weight - used for the interpretability figure."""
        return None if self.last_gate is None else self.last_gate.mean(dim=(0, 2, 3))


# --------------------------------------------------------------------------- FDRM
class HaarDWT(nn.Module):
    """Orthonormal Haar transform as a fixed grouped convolution, exactly
    inverted by the transposed convolution with the same filters."""

    def __init__(self):
        super().__init__()
        h = torch.tensor([[0.5, 0.5], [0.5, 0.5]])
        g1 = torch.tensor([[0.5, 0.5], [-0.5, -0.5]])
        g2 = torch.tensor([[0.5, -0.5], [0.5, -0.5]])
        g3 = torch.tensor([[0.5, -0.5], [-0.5, 0.5]])
        self.register_buffer("filt", torch.stack([h, g1, g2, g3]).unsqueeze(1) * 2.0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:      # [B,C,H,W]->[B,4C,H/2,W/2]
        b, c, h, w = x.shape
        y = F.conv2d(x.reshape(b * c, 1, h, w), self.filt.to(x.dtype), stride=2)
        return y.reshape(b, c * 4, h // 2, w // 2)

    def inverse(self, y: torch.Tensor) -> torch.Tensor:      # [B,4C,H,W]->[B,C,2H,2W]
        b, c4, h, w = y.shape
        c = c4 // 4
        x = F.conv_transpose2d(y.reshape(b * c, 4, h, w), self.filt.to(y.dtype), stride=2)
        return x.reshape(b, c, h * 2, w * 2) * 0.25


class FrequencyDomainRefinement(nn.Module):
    """Wavelet-domain refinement: per-subband processing with learnable
    soft-thresholding (classical wavelet shrinkage, made learnable), plus
    cross-subband mixing, then an exact inverse transform.

    The v1 module was three convolutions of different kernel size and touched no
    frequency representation at all.
    """

    def __init__(self, channels: int):
        super().__init__()
        self.dwt = HaarDWT()
        self.sub = nn.ModuleList([
            nn.Sequential(nn.Conv2d(channels, channels, 3, 1, 1), nn.LeakyReLU(0.1, True),
                          nn.Conv2d(channels, channels, 3, 1, 1))
            for _ in range(4)
        ])
        self.thresh = nn.Parameter(torch.zeros(3, channels))   # detail subbands
        self.mix = nn.Conv2d(channels * 4, channels * 4, 1)
        self.fuse = nn.Conv2d(channels, channels, 3, 1, 1)

    @staticmethod
    def _shrink(x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        t = F.softplus(t)[None, :, None, None].to(x.dtype)
        return torch.sign(x) * torch.clamp(x.abs() - t, min=0.0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        pad_h, pad_w = h % 2, w % 2
        if pad_h or pad_w:
            x = F.pad(x, (0, pad_w, 0, pad_h), mode="reflect")
        bands = self.dwt(x).chunk(4, dim=1)
        out = []
        for i, band in enumerate(bands):
            y = self.sub[i](band)
            if i > 0:                                  # shrink detail subbands only
                y = self._shrink(y, self.thresh[i - 1])
            out.append(y + band)
        y = self.dwt.inverse(self.mix(torch.cat(out, dim=1)))
        if pad_h or pad_w:
            y = y[..., :h, :w]
        return self.fuse(y) + x[..., :h, :w]


# ---------------------------------------------------------------------- upsamplers
class BackProjectionUpsampler(nn.Module):
    """Learned upsampling with iterative observation-model correction:

        y <- Up(x);   then repeatedly   y <- y + Up_res( x - Down(y) )

    The residual is measured in LR space, where the actual observation is
    available, so each step pulls the estimate back onto the observation
    manifold. Bicubic interpolation cannot do this because it never looks at
    how well its own output re-explains the input.
    """

    def __init__(self, bands: int, scale: int, width: int = 64, iters: int = 2):
        super().__init__()
        self.scale, self.iters = scale, iters
        stages, s, ch = [], scale, bands
        while s > 1:
            step = 2 if s % 2 == 0 else s
            stages += [nn.Conv2d(ch, width * step * step, 3, 1, 1),
                       nn.PixelShuffle(step), nn.LeakyReLU(0.1, True)]
            ch = width
            s //= step
        stages += [nn.Conv2d(ch, bands, 3, 1, 1)]
        self.up = nn.Sequential(*stages)
        self.down = nn.Sequential(
            nn.Conv2d(bands, width, 3, 1, 1), nn.LeakyReLU(0.1, True),
            nn.Conv2d(width, width, 2 * scale + 1, scale, scale), nn.LeakyReLU(0.1, True),
            nn.Conv2d(width, bands, 3, 1, 1),
        )
        self.up_res = nn.Sequential(
            nn.Conv2d(bands, width, 3, 1, 1), nn.LeakyReLU(0.1, True),
            nn.Conv2d(width, bands * scale * scale, 3, 1, 1), nn.PixelShuffle(scale),
        )

    def forward(self, lr: torch.Tensor) -> torch.Tensor:
        base = F.interpolate(lr, scale_factor=self.scale, mode="bicubic",
                             align_corners=False)
        y = self.up(lr) + base                     # learned residual over a cheap prior
        for _ in range(self.iters):
            err = lr - self.down(y)
            if err.shape[-2:] != lr.shape[-2:]:
                err = F.interpolate(err, size=lr.shape[-2:], mode="bilinear",
                                    align_corners=False)
            y = y + self.up_res(err)
        return y


class BicubicUpsampler(nn.Module):
    """Control arm for the back-projection ablation."""

    def __init__(self, bands: int, scale: int, width: int = 64):
        super().__init__()
        self.scale = scale
        self.refine = nn.Sequential(
            nn.Conv2d(bands, width, 3, 1, 1), nn.LeakyReLU(0.1, True),
            nn.Conv2d(width, bands, 3, 1, 1),
        )

    def forward(self, lr: torch.Tensor) -> torch.Tensor:
        y = F.interpolate(lr, scale_factor=self.scale, mode="bicubic",
                          align_corners=False)
        return y + self.refine(y)

In [ ]:
%%writefile daetf/model.py
"""DAETF-Net: the assembled network.

Flow:
    (LR-HSI, MSI) -> degradation code -----------------------------+
    LR-HSI -> back-projection upsampler -> coarse HR estimate y0   |
    y0  -> equivariant feature extractor -> FiLM <-----------------+
    MSI -> encoder ----------------------> FiLM <------------------+
    (f_hsi, f_msi) -> Tucker interaction -> region-aware MoE -> wavelet refinement
    -> residual reconstruction, added to y0

Every module can be swapped for a matched control arm through the Config
ablation switches, so each contribution is measured against a like-for-like
baseline rather than against its own absence.
"""

from __future__ import annotations

from typing import Dict, Tuple

import torch
import torch.nn as nn

from .config import Config
from .modules import (BackProjectionUpsampler, BicubicUpsampler, DegradationEncoder,
                      EquivariantFeatureExtractor, FiLM, FrequencyDomainRefinement,
                      PlainFeatureExtractor, RegionAwareMoE,
                      TensorSpectralSpatialEncoder)


class DAETFNet(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        if cfg.bands is None or cfg.msi_bands is None:
            raise ValueError("Config.bands/msi_bands are unset - call cfg.resolve() "
                             "or pass them explicitly before building the model")
        self.cfg = cfg
        c, b, m = cfg.width, cfg.bands, cfg.msi_bands

        self.upsampler = (
            BackProjectionUpsampler(b, cfg.scale, width=c, iters=cfg.bp_iters)
            if cfg.use_backprojection else BicubicUpsampler(b, cfg.scale, width=c)
        )
        self.deg = DegradationEncoder(b, m, code=cfg.code_dim)
        self.efe = (
            EquivariantFeatureExtractor(b, cfg.equi_width, c, depth=cfg.equi_depth)
            if cfg.use_equivariant
            else PlainFeatureExtractor(b, cfg.equi_width, c, depth=cfg.equi_depth)
        )
        self.msi_enc = nn.Sequential(
            nn.Conv2d(m, c, 3, 1, 1), nn.LeakyReLU(0.1, True), nn.Conv2d(c, c, 3, 1, 1),
        )
        self.film_h = FiLM(cfg.code_dim, c)
        self.film_m = FiLM(cfg.code_dim, c)

        self.tsse = (TensorSpectralSpatialEncoder(c, c, c, rank=cfg.rank)
                     if cfg.use_tsse else None)
        self.concat_fuse = None if cfg.use_tsse else nn.Sequential(
            nn.Conv2d(c * 2, c, 1), nn.LeakyReLU(0.1, True), nn.Conv2d(c, c, 3, 1, 1)
        )
        self.moe = (RegionAwareMoE(c, experts=cfg.experts, topk=cfg.topk)
                    if cfg.use_moe else None)
        self.plain_block = None if cfg.use_moe else nn.Sequential(
            nn.Conv2d(c, c, 3, 1, 1), nn.LeakyReLU(0.1, True), nn.Conv2d(c, c, 3, 1, 1)
        )
        self.fdrm = FrequencyDomainRefinement(c) if cfg.use_fdrm else None
        self.recon = nn.Sequential(
            nn.Conv2d(c, c, 3, 1, 1), nn.LeakyReLU(0.1, True), nn.Conv2d(c, b, 3, 1, 1)
        )

    def _trunk(self, lr_hsi: torch.Tensor, msi: torch.Tensor
               ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        code, deg_params = self.deg(lr_hsi, msi)
        if not self.cfg.use_degradation_code:
            code = torch.zeros_like(code)
        y0 = self.upsampler(lr_hsi)
        fh = self.film_h(self.efe(y0), code)
        fm = self.film_m(self.msi_enc(msi), code)
        z = (self.tsse(fh, fm) if self.tsse is not None
             else self.concat_fuse(torch.cat([fh, fm], dim=1)))
        return y0, z, deg_params

    def features(self, lr_hsi: torch.Tensor, msi: torch.Tensor) -> torch.Tensor:
        """Pooled bottleneck features, used by the MMD domain-alignment term."""
        return self._trunk(lr_hsi, msi)[1].mean(dim=(2, 3))

    def forward(self, lr_hsi: torch.Tensor, msi: torch.Tensor) -> Dict[str, torch.Tensor]:
        y0, z, deg_params = self._trunk(lr_hsi, msi)
        z = self.moe(z) if self.moe is not None else self.plain_block(z)
        if self.fdrm is not None:
            z = self.fdrm(z)
        out = y0 + self.recon(z)
        return {"out": out, "coarse": y0, "deg": deg_params, "feat": z.mean(dim=(2, 3))}

    def n_params(self) -> int:
        return sum(p.numel() for p in self.parameters())

In [ ]:
%%writefile daetf/losses.py
"""Spectral-Physical Composite (SPC) loss.

    L = w1 Charbonnier + w2 SAM + w3 gradient + w4 (1 - SSIM)      [fidelity]
      + w5 || Down(Y) - LR ||  + w6 || SRF(Y) - MSI ||             [physics]
      + w7 MoE-balance + w8 ||G||_*  + w9 degradation-regression   [regularisers]
      + w10 MMD(source, target)                                    [domain]

Two properties matter for the research claim:

1. The SAM term optimises the metric on which every benchmarked baseline
   degrades under domain shift (SAM 2-7 deg in-domain vs 8-36 deg cross-domain),
   rather than optimising only the metric that already looks good.

2. The two physics terms need no ground truth. They are computable on any
   unseen scene, which is precisely what makes self-supervised test-time
   adaptation possible on a new sensor or dataset.
"""

from __future__ import annotations

from typing import Dict, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

from .config import Config
from .degrade import FixedDegradation
from .metrics import ssim_torch


def charbonnier(x: torch.Tensor, y: torch.Tensor, eps: float = 1e-3) -> torch.Tensor:
    """Robust L1: differentiable at zero, less outlier-sensitive than L2."""
    return torch.sqrt((x - y) ** 2 + eps ** 2).mean()


def sam_loss(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """Mean spectral angle in radians."""
    p, t = pred.flatten(2), target.flatten(2)
    num = (p * t).sum(dim=1)
    den = p.norm(dim=1) * t.norm(dim=1)
    cos = (num / den.clamp_min(eps)).clamp(-1 + 1e-6, 1 - 1e-6)
    return torch.acos(cos).mean()


def gradient_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    dx_p = pred[..., :, 1:] - pred[..., :, :-1]
    dx_t = target[..., :, 1:] - target[..., :, :-1]
    dy_p = pred[..., 1:, :] - pred[..., :-1, :]
    dy_t = target[..., 1:, :] - target[..., :-1, :]
    return F.l1_loss(dx_p, dx_t) + F.l1_loss(dy_p, dy_t)


def mmd_rbf(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """Multi-bandwidth RBF maximum mean discrepancy, with the bandwidth set from
    the median pairwise distance so it adapts to the feature scale."""
    z = torch.cat([x, y], dim=0)
    d = torch.cdist(z, z) ** 2
    n = x.shape[0]
    med = d.detach().flatten().median().clamp_min(1e-6)
    k = sum(torch.exp(-d / (med * s)) for s in (0.25, 0.5, 1.0, 2.0, 4.0))
    return k[:n, :n].mean() + k[n:, n:].mean() - 2 * k[:n, n:].mean()


class SPCLoss(nn.Module):
    def __init__(self, cfg: Config, srf: torch.Tensor):
        super().__init__()
        self.cfg = cfg
        self.degrade = FixedDegradation.from_config(cfg)
        self.register_buffer("srf", srf)      # [bands, msi_bands]

    def apply_srf(self, x: torch.Tensor) -> torch.Tensor:
        """Project a hyperspectral cube through the recovered spectral response."""
        w = self.srf.to(x.dtype).t().reshape(self.srf.shape[1], self.srf.shape[0], 1, 1)
        return F.conv2d(x, w)

    def forward(self, out: Dict[str, torch.Tensor], target: torch.Tensor,
                lr_hsi: torch.Tensor, msi: torch.Tensor, model,
                deg_gt: Optional[torch.Tensor] = None,
                kernel: Optional[torch.Tensor] = None,
                tgt_feat: Optional[torch.Tensor] = None,
                supervised: bool = True) -> Tuple[torch.Tensor, Dict[str, float]]:
        cfg = self.cfg
        pred = out["out"]
        logs: Dict[str, float] = {}
        total = pred.new_zeros(())

        if supervised:
            l_char = charbonnier(pred, target)
            l_sam = sam_loss(pred, target)
            l_grad = gradient_loss(pred, target)
            l_ssim = 1.0 - ssim_torch(pred.clamp(0, 1).float(), target.float())
            total = (total + cfg.w_char * l_char + cfg.w_sam * l_sam
                     + cfg.w_grad * l_grad + cfg.w_ssim * l_ssim)
            logs.update(char=l_char.item(), sam=l_sam.item(),
                        grad=l_grad.item(), ssim=l_ssim.item())

        # --- physics: valid on any domain, with or without ground truth -------
        if cfg.use_physics or not supervised:
            l_spat = charbonnier(self.degrade(pred, kernel), lr_hsi)
            l_spec = charbonnier(self.apply_srf(pred), msi)
            total = total + cfg.w_spat * l_spat + cfg.w_spec * l_spec
            logs.update(spat=l_spat.item(), spec=l_spec.item())

        # --- regularisers -------------------------------------------------------
        if getattr(model, "moe", None) is not None:
            l_bal = model.moe.balance_loss()
            total = total + cfg.w_bal * l_bal
            logs["bal"] = l_bal.item()

        if supervised:
            if getattr(model, "tsse", None) is not None:
                l_rank = model.tsse.rank_penalty()
                total = total + cfg.w_rank * l_rank
                logs["rank"] = l_rank.item()
            if deg_gt is not None:
                l_deg = F.smooth_l1_loss(out["deg"].float(), deg_gt.float())
                total = total + cfg.w_deg * l_deg
                logs["deg"] = l_deg.item()
            if tgt_feat is not None:
                l_mmd = mmd_rbf(out["feat"].float(), tgt_feat.float())
                total = total + cfg.w_mmd * l_mmd
                logs["mmd"] = l_mmd.item()

        logs["total"] = total.item()
        return total, logs

In [ ]:
%%writefile daetf/data.py
"""Datasets, scene caching and spectral-response estimation.

The v1 dataset returned `torch.randn(...)` with a hardcoded length of 100, so
nothing was ever trained on real data. This module loads the actual .mat scenes
and synthesises the observation pair through the physical forward model.
"""

from __future__ import annotations

import math
import random
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset

from .config import Config
from .degrade import blur_downsample, gaussian_kernel2d
from .io_utils import find_pairs, load_mat, to_chw01


class SceneCache:
    """Bounded LRU cache of decoded scenes, held in float16.

    Harvard scenes are 1040x1392x31; caching them all as float32 would need
    ~5.4 GB, so scenes are stored halved and evicted least-recently-used.
    """

    def __init__(self, bands: int, msi_bands: int, limit: int = 12):
        self.bands, self.msi_bands, self.limit = bands, msi_bands, limit
        self.store: Dict[str, Tuple[np.ndarray, np.ndarray]] = {}
        self.order: List[str] = []

    def get(self, stem: str, hsi_path: str, rgb_path: str
            ) -> Tuple[np.ndarray, np.ndarray]:
        if stem in self.store:
            self.order.remove(stem)
            self.order.append(stem)
            return self.store[stem]
        hsi = to_chw01(load_mat(hsi_path), self.bands)
        rgb = to_chw01(load_mat(rgb_path), self.msi_bands)
        if rgb.shape[-2:] != hsi.shape[-2:]:
            t = torch.from_numpy(rgb)[None]
            rgb = F.interpolate(t, size=hsi.shape[-2:], mode="bicubic",
                                align_corners=False).clamp(0, 1)[0].numpy()
        item = (hsi.astype(np.float16), rgb.astype(np.float16))
        self.store[stem] = item
        self.order.append(stem)
        while len(self.order) > self.limit:
            self.store.pop(self.order.pop(0), None)
        return item


class FusionPatchDataset(Dataset):
    """Samples HR patches and synthesises (LR-HSI, MSI) through the forward
    model, randomising blur, noise and spectral response.

    The randomisation is the domain-shift defence: a model that has only ever
    seen one fixed bicubic degradation has no reason to work on a real sensor.
    """

    def __init__(self, root: str, split: str, cfg: Config, train: bool = True,
                 srf: Optional[np.ndarray] = None, length: int = 8000):
        self.cfg, self.train, self.length = cfg, train, length
        self.pairs = find_pairs(root, split)
        self.cache = SceneCache(cfg.bands, cfg.msi_bands,
                                limit=len(self.pairs) if train else 4)
        self.srf = srf

    def __len__(self) -> int:
        return self.length if self.train else len(self.pairs)

    def _sample_kernel(self) -> Tuple[torch.Tensor, List[float]]:
        cfg = self.cfg
        sx = random.uniform(*cfg.sigma_range)
        sy = sx if random.random() > cfg.aniso else random.uniform(*cfg.sigma_range)
        th = random.uniform(0, math.pi)
        return (gaussian_kernel2d(cfg.blur_ksize, sx, sy, th),
                [sx, sy, math.sin(2 * th), math.cos(2 * th)])

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        cfg = self.cfg
        stem, hp, rp = (self.pairs[random.randrange(len(self.pairs))] if self.train
                        else self.pairs[idx % len(self.pairs)])
        hsi, rgb = self.cache.get(stem, hp, rp)

        if self.train:
            p = cfg.patch
            _, h, w = hsi.shape
            top, left = random.randrange(0, h - p + 1), random.randrange(0, w - p + 1)
            gt = torch.from_numpy(hsi[:, top:top + p, left:left + p].astype(np.float32))
            msi = torch.from_numpy(rgb[:, top:top + p, left:left + p].astype(np.float32))
            if random.random() < 0.5:
                gt, msi = torch.flip(gt, [-1]), torch.flip(msi, [-1])
            k = random.randrange(4)          # the p4 stem handles these natively
            if k:
                gt, msi = torch.rot90(gt, k, (-2, -1)), torch.rot90(msi, k, (-2, -1))
        else:
            p = (min(hsi.shape[1], hsi.shape[2]) // cfg.scale) * cfg.scale
            gt = torch.from_numpy(hsi[:, :p, :p].astype(np.float32))
            msi = torch.from_numpy(rgb[:, :p, :p].astype(np.float32))

        es = cfg.eval_sigma
        kernel, deg = (self._sample_kernel() if self.train else
                       (gaussian_kernel2d(cfg.blur_ksize, es, es, 0.0), [es, es, 0.0, 1.0]))

        lr = blur_downsample(gt[None], kernel, cfg.scale)[0]
        noise = random.uniform(*cfg.noise_range) if self.train else 0.0
        if noise > 0:
            lr = (lr + torch.randn_like(lr) * noise).clamp(0, 1)

        # sometimes replace the real RGB with a jittered synthetic MSI, so the
        # model never assumes one fixed spectral response function
        if self.train and self.srf is not None and random.random() < cfg.srf_jitter:
            s = torch.from_numpy(self.srf).float()
            s = (s * (1 + 0.15 * torch.randn_like(s))).clamp_min(0)
            s = s / s.sum(0, keepdim=True).clamp_min(1e-6) * float(self.srf.sum(0).mean())
            msi = torch.einsum("chw,cm->mhw", gt, s).clamp(0, 1)

        return {"lr": lr, "msi": msi, "gt": gt,
                "deg": torch.tensor(deg + [noise], dtype=torch.float32),
                "kernel": kernel, "name": stem}


def estimate_srf(root: str, split: str, cfg: Config, max_scenes: int = 8,
                 samples_per_scene: int = 20000) -> np.ndarray:
    """Least-squares spectral response function: min_S || HSI @ S - RGB ||^2.

    Recovering the SRF from the data makes the spectral-consistency loss a real
    physical constraint instead of a hand-picked approximation, and it adapts
    automatically to a dataset whose RGB was rendered with a different response.
    """
    pairs = find_pairs(root, split)[:max_scenes]
    xs, ys = [], []
    rng = np.random.default_rng(0)
    for stem, hp, rp in pairs:
        hsi = to_chw01(load_mat(hp), cfg.bands)
        rgb = to_chw01(load_mat(rp), cfg.msi_bands)
        if rgb.shape[-2:] != hsi.shape[-2:]:
            rgb = F.interpolate(torch.from_numpy(rgb)[None], size=hsi.shape[-2:],
                                mode="bicubic", align_corners=False)[0].numpy()
        h = hsi.reshape(cfg.bands, -1).T
        r = rgb.reshape(cfg.msi_bands, -1).T
        idx = rng.choice(h.shape[0], size=min(samples_per_scene, h.shape[0]),
                         replace=False)
        xs.append(h[idx])
        ys.append(r[idx])
    x = np.concatenate(xs).astype(np.float64)
    y = np.concatenate(ys).astype(np.float64)
    s, *_ = np.linalg.lstsq(x, y, rcond=None)
    return np.clip(s, 0.0, None).astype(np.float32)

In [ ]:
%%writefile daetf/engine.py
"""Training, full-scene inference, evaluation and test-time adaptation."""

from __future__ import annotations

import json
import math
import os
import random
import time
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from .config import Config
from .data import FusionPatchDataset, SceneCache, estimate_srf
from .degrade import FixedDegradation
from .io_utils import find_pairs
from .losses import SPCLoss
from .metrics import evaluate_arrays
from .model import DAETFNet


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def cosine_lr(step: int, cfg: Config) -> float:
    if step < cfg.warmup:
        return cfg.lr * step / max(cfg.warmup, 1)
    t = (step - cfg.warmup) / max(cfg.iters - cfg.warmup, 1)
    return cfg.min_lr + 0.5 * (cfg.lr - cfg.min_lr) * (1 + math.cos(math.pi * t))


# ------------------------------------------------------------------- inference
@torch.no_grad()
def tiled_inference(model: DAETFNet, lr: torch.Tensor, msi: torch.Tensor, scale: int,
                    tile_hr: int = 256, overlap: int = 32) -> torch.Tensor:
    """Hann-weighted overlapping tiles, so full 512x512 and 1040x1392 scenes fit
    in 16 GB without seams appearing at tile boundaries."""
    model.eval()
    bsz, _, h_hr, w_hr = msi.shape
    tile_lr, ov_lr = tile_hr // scale, overlap // scale
    tile_lr = min(tile_lr, lr.shape[2], lr.shape[3])
    tile_hr = tile_lr * scale
    step_lr = max(tile_lr - ov_lr, 1)
    out = torch.zeros(bsz, model.cfg.bands, h_hr, w_hr, device=lr.device, dtype=torch.float32)
    wsum = torch.zeros(bsz, 1, h_hr, w_hr, device=lr.device, dtype=torch.float32)

    win1d = torch.hann_window(tile_hr, periodic=False, device=lr.device).clamp_min(1e-3)
    win = (win1d[:, None] * win1d[None, :])[None, None]

    ys = list(range(0, max(lr.shape[2] - tile_lr, 0) + 1, step_lr))
    xs = list(range(0, max(lr.shape[3] - tile_lr, 0) + 1, step_lr))
    if ys[-1] + tile_lr < lr.shape[2]:
        ys.append(lr.shape[2] - tile_lr)
    if xs[-1] + tile_lr < lr.shape[3]:
        xs.append(lr.shape[3] - tile_lr)

    for y0 in ys:
        for x0 in xs:
            y1, x1 = y0 + tile_lr, x0 + tile_lr
            hy0, hx0, hy1, hx1 = y0 * scale, x0 * scale, y1 * scale, x1 * scale
            pred = model(lr[:, :, y0:y1, x0:x1], msi[:, :, hy0:hy1, hx0:hx1])["out"].float()
            w = win[..., :pred.shape[-2], :pred.shape[-1]]
            out[:, :, hy0:hy1, hx0:hx1] += pred * w
            wsum[:, :, hy0:hy1, hx0:hx1] += w
    return (out / wsum.clamp_min(1e-6)).clamp(0, 1)


@torch.no_grad()
def evaluate_dataset(model: DAETFNet, root: str, cfg: Config, split: str = "Test",
                     device: str = "cuda", limit: Optional[int] = None,
                     tile_hr: int = 256, verbose: bool = True,
                     return_rows: bool = False):
    """Full-scene evaluation through the unified metric module.

    With return_rows=True the per-scene table is returned as well, which is what
    the paired significance tests operate on.
    """
    pairs = find_pairs(root, split)
    if limit:
        pairs = pairs[:limit]
    cache = SceneCache(cfg.bands, cfg.msi_bands, limit=2)
    degrade = FixedDegradation.from_config(cfg).to(device)
    rows, agg = [], {"psnr": [], "ssim": [], "sam": [], "ergas": []}

    for stem, hp, rp in pairs:
        hsi, rgb = cache.get(stem, hp, rp)
        h = (hsi.shape[1] // cfg.scale) * cfg.scale
        w = (hsi.shape[2] // cfg.scale) * cfg.scale
        gt = torch.from_numpy(hsi[:, :h, :w].astype(np.float32))[None].to(device)
        msi = torch.from_numpy(rgb[:, :h, :w].astype(np.float32))[None].to(device)
        lr = degrade(gt)
        pred = tiled_inference(model, lr, msi, cfg.scale, tile_hr=tile_hr)
        m = evaluate_arrays(pred[0].cpu().numpy().transpose(1, 2, 0),
                            gt[0].cpu().numpy().transpose(1, 2, 0), cfg.scale)
        rows.append((stem, m))
        for k, v in m.items():
            agg[k].append(v)
        if verbose:
            print(f"  {stem:<24} PSNR={m['psnr']:7.3f}  SSIM={m['ssim']:.4f}  "
                  f"SAM={m['sam']:6.3f}  ERGAS={m['ergas']:8.3f}")
        del gt, msi, lr, pred
        if device == "cuda":
            torch.cuda.empty_cache()

    mean = {k: float(np.mean(v)) for k, v in agg.items()}
    if verbose:
        print(f"  {'MEAN':<24} PSNR={mean['psnr']:7.3f}  SSIM={mean['ssim']:.4f}  "
              f"SAM={mean['sam']:6.3f}  ERGAS={mean['ergas']:8.3f}")
    if return_rows:
        return mean, [{"scene": s, **m} for s, m in rows]
    return mean


# -------------------------------------------------------------------- training
def train(cfg: Config, device: str = "cuda", align_target: bool = True,
          log_fn=print) -> Tuple[DAETFNet, Dict]:
    """Train on the source domain.

    When `align_target` is set and a target root is configured, unlabelled
    target patches are drawn alongside and aligned with an MMD penalty. No
    ground truth from the target domain is ever used, so the cross-domain
    evaluation stays honest.
    """
    cfg.resolve(verbose=False)          # idempotent: fills only what is None
    set_seed(cfg.seed)
    os.makedirs(cfg.out_dir, exist_ok=True)

    log_fn("estimating SRF from the training pairs ...")
    srf = estimate_srf(cfg.source_root, "Train", cfg)
    log_fn(f"SRF shape {srf.shape}, column sums {srf.sum(0).round(3).tolist()}")

    train_set = FusionPatchDataset(cfg.source_root, "Train", cfg, train=True, srf=srf,
                                   length=cfg.iters * cfg.batch)
    loader = DataLoader(train_set, batch_size=cfg.batch, shuffle=False,
                        num_workers=cfg.workers, pin_memory=(device == "cuda"),
                        drop_last=True, persistent_workers=cfg.workers > 0)

    tgt_loader = None
    if align_target and cfg.target_root:
        tgt_set = FusionPatchDataset(cfg.target_root, "Train", cfg, train=True, srf=srf,
                                     length=cfg.iters * cfg.batch)
        tgt_loader = iter(DataLoader(tgt_set, batch_size=cfg.batch, shuffle=False,
                                     num_workers=max(1, cfg.workers // 2), drop_last=True))
        log_fn(f"domain alignment enabled against {cfg.target_root} (unlabelled)")

    model = DAETFNet(cfg).to(device)
    crit = SPCLoss(cfg, torch.from_numpy(srf)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=1e-5,
                            betas=(0.9, 0.99))
    use_amp = cfg.amp and device == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    n_params = model.n_params()
    log_fn(f"DAETF-Net v2: {n_params / 1e6:.2f} M parameters")

    history: Dict[str, list] = {"iter": [], "loss": [], "val": [], "cfg": cfg.to_dict()}
    best, t0 = -1e9, time.time()

    model.train()
    for step, batch in enumerate(loader, start=1):
        if step > cfg.iters:
            break
        for g in opt.param_groups:
            g["lr"] = cosine_lr(step, cfg)

        lr_hsi = batch["lr"].to(device, non_blocking=True)
        msi = batch["msi"].to(device, non_blocking=True)
        gt = batch["gt"].to(device, non_blocking=True)
        deg_gt = batch["deg"].to(device, non_blocking=True)
        kernel = batch["kernel"].to(device, non_blocking=True)

        tgt_feat = None
        if tgt_loader is not None:
            tb = next(tgt_loader)
            with torch.amp.autocast("cuda", enabled=use_amp):
                tgt_feat = model.features(tb["lr"].to(device), tb["msi"].to(device))

        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=use_amp):
            out = model(lr_hsi, msi)
            loss, logs = crit(out, gt, lr_hsi, msi, model, deg_gt=deg_gt,
                              kernel=kernel, tgt_feat=tgt_feat)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(opt)
        scaler.update()

        if step % cfg.log_every == 0:
            rate = step / (time.time() - t0)
            eta = (cfg.iters - step) / max(rate, 1e-6) / 60
            log_fn(f"it {step:6d}/{cfg.iters}  loss {logs['total']:.4f}  "
                   f"char {logs.get('char', 0):.4f}  sam {logs.get('sam', 0):.4f}  "
                   f"spat {logs.get('spat', 0):.4f}  spec {logs.get('spec', 0):.4f}  "
                   f"lr {opt.param_groups[0]['lr']:.2e}  {rate:.2f} it/s  eta {eta:.0f}m")
            history["iter"].append(step)
            history["loss"].append(logs["total"])

        if step % cfg.val_every == 0 or step == cfg.iters:
            m = evaluate_dataset(model, cfg.source_root, cfg, "Test", device,
                                 limit=cfg.val_scenes, verbose=False)
            log_fn(f"  [val@{step}] PSNR {m['psnr']:.3f}  SAM {m['sam']:.3f}  "
                   f"ERGAS {m['ergas']:.3f}")
            history["val"].append({"iter": step, **m})
            if m["psnr"] > best:
                best = m["psnr"]
                torch.save({"model": model.state_dict(), "cfg": cfg.to_dict(),
                            "srf": srf, "val": m}, os.path.join(cfg.out_dir, "daetf_best.pth"))
            model.train()

    torch.save({"model": model.state_dict(), "cfg": cfg.to_dict(), "srf": srf,
                "params": n_params}, os.path.join(cfg.out_dir, "daetf_final.pth"))
    with open(os.path.join(cfg.out_dir, "history.json"), "w") as f:
        json.dump(history, f, indent=1)
    return model, history


# ------------------------------------------------------- test-time adaptation
@torch.no_grad()
def _clone_state(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {k: v.detach().clone() for k, v in model.state_dict().items()}


def test_time_adapt(model: DAETFNet, lr: torch.Tensor, msi: torch.Tensor,
                    crit: SPCLoss, steps: int = 30, lr_rate: float = 5e-5,
                    restore: bool = True) -> torch.Tensor:
    """Self-supervised adaptation on a single unlabelled target scene.

    Only the physics terms are used - they need no ground truth - so this runs
    on a new dataset or sensor exactly as it would in deployment. Only the
    conditioning-related parameters are adapted, which keeps it stable and cheap.
    """
    state = _clone_state(model) if restore else None
    model.train()
    params = [p for n, p in model.named_parameters()
              if any(k in n for k in ("deg", "film", "moe.gate", "fdrm"))]
    opt = torch.optim.Adam(params, lr=lr_rate)
    for _ in range(steps):
        opt.zero_grad(set_to_none=True)
        out = model(lr, msi)
        loss, _ = crit(out, out["out"].detach(), lr, msi, model, supervised=False)
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        pred = model(lr, msi)["out"].clamp(0, 1)
    if state is not None:
        model.load_state_dict(state)
    return pred


@torch.no_grad()
def evaluate_with_tta(model: DAETFNet, root: str, cfg: Config, srf: np.ndarray,
                      split: str = "Test", device: str = "cuda",
                      steps: int = 30, limit: Optional[int] = None,
                      tile_hr: int = 256, verbose: bool = True):
    """Cross-domain evaluation where each scene is adapted before scoring.

    Every scene starts from the same trained weights. Adapting cumulatively
    across scenes would make the result depend on the order the scenes happen
    to be listed in, and would quietly let information leak from one test scene
    into the next.
    """
    crit = SPCLoss(cfg, torch.from_numpy(srf)).to(device)
    pairs = find_pairs(root, split)
    if limit:
        pairs = pairs[:limit]
    cache = SceneCache(cfg.bands, cfg.msi_bands, limit=2)
    degrade = FixedDegradation.from_config(cfg).to(device)
    rows, agg = [], {"psnr": [], "ssim": [], "sam": [], "ergas": []}
    base_state = _clone_state(model)          # restored before every scene

    for stem, hp, rp in pairs:
        model.load_state_dict(base_state)
        hsi, rgb = cache.get(stem, hp, rp)
        h = (hsi.shape[1] // cfg.scale) * cfg.scale
        w = (hsi.shape[2] // cfg.scale) * cfg.scale
        # adapt on a centre crop to bound memory, then infer over the full scene
        ch, cw = min(h, tile_hr * 2), min(w, tile_hr * 2)
        oy, ox = (h - ch) // 2, (w - cw) // 2
        gt = torch.from_numpy(hsi[:, :h, :w].astype(np.float32))[None].to(device)
        msi = torch.from_numpy(rgb[:, :h, :w].astype(np.float32))[None].to(device)
        lr = degrade(gt)
        crop_lr = lr[:, :, oy // cfg.scale: (oy + ch) // cfg.scale,
                     ox // cfg.scale: (ox + cw) // cfg.scale]
        crop_msi = msi[:, :, oy:oy + ch, ox:ox + cw]
        with torch.enable_grad():
            test_time_adapt(model, crop_lr, crop_msi, crit, steps=steps, restore=False)
        pred = tiled_inference(model, lr, msi, cfg.scale, tile_hr=tile_hr)
        m = evaluate_arrays(pred[0].cpu().numpy().transpose(1, 2, 0),
                            gt[0].cpu().numpy().transpose(1, 2, 0), cfg.scale)
        rows.append({"scene": stem, **m})
        for k, v in m.items():
            agg[k].append(v)
        if verbose:
            print(f"  {stem:<24} PSNR={m['psnr']:7.3f}  SSIM={m['ssim']:.4f}  "
                  f"SAM={m['sam']:6.3f}  ERGAS={m['ergas']:8.3f}")
        del gt, msi, lr, pred
        if device == "cuda":
            torch.cuda.empty_cache()

    model.load_state_dict(base_state)         # leave the caller's model untouched
    mean = {k: float(np.mean(v)) for k, v in agg.items()}
    if verbose:
        print(f"  {'MEAN (TTA)':<24} PSNR={mean['psnr']:7.3f}  SSIM={mean['ssim']:.4f}  "
              f"SAM={mean['sam']:6.3f}  ERGAS={mean['ergas']:8.3f}")
    return mean, rows


def load_checkpoint(path: str, device: str = "cuda") -> Tuple[DAETFNet, Config, np.ndarray]:
    """Rebuild a model from a checkpoint without needing the original Config."""
    ck = torch.load(path, map_location=device, weights_only=False)
    cfg = Config(**ck["cfg"])
    model = DAETFNet(cfg).to(device)
    model.load_state_dict(ck["model"])
    model.eval()
    return model, cfg, ck["srf"]

In [ ]:
%%writefile daetf/experiments.py
"""Experiment harness: ablations, efficiency profiling, statistics and tables.

What separates a Q1 submission from a demo is not the architecture, it is the
evidence around it. This module produces:

  * per-scene results, not just means, so comparisons can be paired
  * a paired Wilcoxon signed-rank test and Cohen's d against every baseline
  * bootstrap 95% confidence intervals on each mean
  * a component ablation with matched control arms
  * multi-seed repeats, reported as mean +/- std
  * cost accounting: parameters, GFLOPs, latency, peak GPU memory
  * cross-domain transfer with and without test-time adaptation
  * scale-factor generalisation
  * Markdown and LaTeX tables ready to paste into a manuscript

Dependencies are numpy/torch only; the statistics are implemented directly so
the module runs on a bare Kaggle image without scipy.stats.
"""

from __future__ import annotations

import copy
import json
import os
import time
from dataclasses import replace
from typing import Callable, Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn

from .config import Config
from .engine import (evaluate_dataset, evaluate_with_tta, load_checkpoint,
                     set_seed, tiled_inference, train)
from .model import DAETFNet

METRICS = ("psnr", "ssim", "sam", "ergas")
HIGHER_IS_BETTER = {"psnr": True, "ssim": True, "sam": False, "ergas": False}


# ---------------------------------------------------------------- statistics
def bootstrap_ci(values: Sequence[float], n_boot: int = 10000, alpha: float = 0.05,
                 seed: int = 0) -> Tuple[float, float]:
    """Percentile bootstrap confidence interval for the mean."""
    v = np.asarray(values, dtype=np.float64)
    if v.size < 2:
        return (float(v.mean()) if v.size else float("nan"),) * 2
    rng = np.random.default_rng(seed)
    means = rng.choice(v, size=(n_boot, v.size), replace=True).mean(axis=1)
    return float(np.percentile(means, 100 * alpha / 2)), \
        float(np.percentile(means, 100 * (1 - alpha / 2)))


def _normal_sf(z: float) -> float:
    """Upper-tail standard normal probability via the error function."""
    return 0.5 * math_erfc(z / (2 ** 0.5))


def math_erfc(x: float) -> float:
    import math
    return math.erfc(x)


def wilcoxon_signed_rank(a: Sequence[float], b: Sequence[float]) -> Dict[str, float]:
    """Paired Wilcoxon signed-rank test with a normal approximation.

    Paired over scenes: every method is scored on the same scenes, so pairing is
    the correct design and is far more sensitive than an unpaired test on the
    10-20 scenes these datasets provide.
    """
    a, b = np.asarray(a, dtype=np.float64), np.asarray(b, dtype=np.float64)
    d = a - b
    d = d[d != 0]
    n = d.size
    if n < 1:
        return {"n": 0, "W": float("nan"), "z": float("nan"), "p": float("nan")}
    order = np.argsort(np.abs(d))
    ranks = np.empty(n, dtype=np.float64)
    ranks[order] = np.arange(1, n + 1)
    # average ranks within ties of |d|
    absd = np.abs(d)[order]
    i = 0
    while i < n:
        j = i
        while j + 1 < n and absd[j + 1] == absd[i]:
            j += 1
        if j > i:
            ranks[order[i:j + 1]] = np.mean(np.arange(i + 1, j + 2))
        i = j + 1
    w_pos = ranks[d > 0].sum()
    w_neg = ranks[d < 0].sum()
    w = min(w_pos, w_neg)
    mu = n * (n + 1) / 4.0
    sigma = (n * (n + 1) * (2 * n + 1) / 24.0) ** 0.5
    z = (w - mu) / sigma if sigma > 0 else 0.0
    p = 2 * _normal_sf(abs(z))
    return {"n": float(n), "W": float(w), "z": float(z), "p": float(min(p, 1.0))}


def cohens_d(a: Sequence[float], b: Sequence[float]) -> float:
    """Paired Cohen's d (mean difference over the sd of the differences)."""
    d = np.asarray(a, dtype=np.float64) - np.asarray(b, dtype=np.float64)
    sd = d.std(ddof=1)
    return float(d.mean() / sd) if sd > 0 else float("inf") if d.mean() else 0.0


def compare_methods(ours: List[Dict], theirs: List[Dict],
                    name_a: str = "ours", name_b: str = "baseline") -> Dict:
    """Paired comparison over the scenes both methods were scored on."""
    by_b = {r["scene"]: r for r in theirs}
    shared = [r for r in ours if r["scene"] in by_b]
    out = {"name_a": name_a, "name_b": name_b, "n_scenes": len(shared)}
    for m in METRICS:
        a = [r[m] for r in shared]
        b = [by_b[r["scene"]][m] for r in shared]
        test = wilcoxon_signed_rank(a, b)
        delta = float(np.mean(a) - np.mean(b))
        improved = delta > 0 if HIGHER_IS_BETTER[m] else delta < 0
        out[m] = {"mean_a": float(np.mean(a)), "mean_b": float(np.mean(b)),
                  "delta": delta, "improved": bool(improved),
                  "p": test["p"], "z": test["z"], "d": cohens_d(a, b),
                  "ci_a": bootstrap_ci(a), "ci_b": bootstrap_ci(b)}
    return out


def summarise_rows(rows: List[Dict]) -> Dict[str, Dict[str, float]]:
    """Mean, std and bootstrap CI for each metric across scenes."""
    out = {}
    for m in METRICS:
        v = [r[m] for r in rows]
        lo, hi = bootstrap_ci(v)
        out[m] = {"mean": float(np.mean(v)), "std": float(np.std(v, ddof=1)) if len(v) > 1 else 0.0,
                  "ci_lo": lo, "ci_hi": hi, "n": len(v)}
    return out


# ------------------------------------------------------------------ efficiency
def count_flops(model: nn.Module, lr_shape: Tuple[int, ...],
                msi_shape: Tuple[int, ...], device: str = "cpu") -> float:
    """Multiply-accumulate count for conv and linear layers, in GFLOPs.

    Implemented with forward hooks rather than an external dependency, so the
    number can be reported from a bare Kaggle image. Counts 2 FLOPs per MAC.
    """
    total = [0]

    def conv_hook(m, inp, out):
        out_elems = out.numel()
        k = m.weight.shape[2] * m.weight.shape[3]
        total[0] += 2 * out_elems * (m.in_channels // m.groups) * k

    def deconv_hook(m, inp, out):
        k = m.weight.shape[2] * m.weight.shape[3]
        total[0] += 2 * inp[0].numel() * (m.out_channels // m.groups) * k

    def lin_hook(m, inp, out):
        total[0] += 2 * out.numel() * m.in_features

    handles = []
    for mod in model.modules():
        if isinstance(mod, nn.Conv2d):
            handles.append(mod.register_forward_hook(conv_hook))
        elif isinstance(mod, nn.ConvTranspose2d):
            handles.append(mod.register_forward_hook(deconv_hook))
        elif isinstance(mod, nn.Linear):
            handles.append(mod.register_forward_hook(lin_hook))

    model.eval()
    with torch.no_grad():
        model(torch.zeros(*lr_shape, device=device), torch.zeros(*msi_shape, device=device))
    for h in handles:
        h.remove()
    return total[0] / 1e9


@torch.no_grad()
def profile_model(model: DAETFNet, cfg: Config, device: str = "cuda",
                  hr: int = 512, warmup: int = 3, runs: int = 10) -> Dict[str, float]:
    """Parameters, GFLOPs, latency and peak memory for one full scene."""
    lr_shape = (1, cfg.bands, hr // cfg.scale, hr // cfg.scale)
    msi_shape = (1, cfg.msi_bands, hr, hr)
    gflops = count_flops(copy.deepcopy(model).to("cpu"), lr_shape, msi_shape, "cpu")

    model = model.to(device).eval()
    lr = torch.zeros(*lr_shape, device=device)
    msi = torch.zeros(*msi_shape, device=device)
    for _ in range(warmup):
        tiled_inference(model, lr, msi, cfg.scale)
    if device == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for _ in range(runs):
        tiled_inference(model, lr, msi, cfg.scale)
    if device == "cuda":
        torch.cuda.synchronize()
    dt = (time.time() - t0) / runs
    peak = torch.cuda.max_memory_allocated() / 2 ** 20 if device == "cuda" else float("nan")
    return {"params_M": model.n_params() / 1e6, "gflops": gflops,
            "latency_s": dt, "peak_mem_MB": peak, "hr": hr}


# ------------------------------------------------------------------- ablations
ABLATIONS: List[Tuple[str, Dict]] = [
    ("full model", {}),
    ("w/o equivariant EFE", {"use_equivariant": False}),
    ("w/o Tucker TSSE", {"use_tsse": False}),
    ("w/o region-aware MoE", {"use_moe": False}),
    ("w/o wavelet FDRM", {"use_fdrm": False}),
    ("w/o back-projection", {"use_backprojection": False}),
    ("w/o degradation code", {"use_degradation_code": False}),
    ("w/o physics losses", {"use_physics": False}),
]


def run_ablation(base_cfg: Config, device: str = "cuda", iters: Optional[int] = None,
                 variants: Optional[List[Tuple[str, Dict]]] = None,
                 log_fn=print) -> List[Dict]:
    """Train each variant from scratch and evaluate in-domain and cross-domain.

    Every variant is trained with an identical budget, seed and data order, so
    differences are attributable to the component rather than to the schedule.
    """
    variants = variants or ABLATIONS
    results = []
    for name, overrides in variants:
        cfg = replace(base_cfg, **overrides)
        if iters:
            cfg.iters = iters
        cfg.out_dir = os.path.join(base_cfg.out_dir, "ablation",
                                   name.replace("/", "").replace(" ", "_"))
        log_fn(f"\n=== ablation: {name} ===")
        model, _ = train(cfg, device=device, log_fn=log_fn)
        row: Dict = {"variant": name, "params_M": model.n_params() / 1e6}
        src, src_rows = evaluate_dataset(model, cfg.source_root, cfg, "Test", device,
                                         verbose=False, return_rows=True)
        row["source"] = src
        row["source_rows"] = src_rows
        if cfg.target_root:
            tgt, tgt_rows = evaluate_dataset(model, cfg.target_root, cfg, "Test", device,
                                             verbose=False, return_rows=True)
            row["target"] = tgt
            row["target_rows"] = tgt_rows
        results.append(row)
        log_fn(f"  {name}: in-domain PSNR {src['psnr']:.3f} SAM {src['sam']:.3f}"
               + (f" | cross-domain PSNR {row['target']['psnr']:.3f} "
                  f"SAM {row['target']['sam']:.3f}" if cfg.target_root else ""))
        del model
        if device == "cuda":
            torch.cuda.empty_cache()
    return results


def run_multiseed(base_cfg: Config, seeds: Sequence[int] = (0, 1, 2),
                  device: str = "cuda", log_fn=print) -> Dict:
    """Repeat the full training run across seeds and report mean +/- std.

    A single run is not evidence; reviewers ask for variance.
    """
    runs = []
    for s in seeds:
        cfg = replace(base_cfg, seed=int(s))
        cfg.out_dir = os.path.join(base_cfg.out_dir, f"seed{s}")
        log_fn(f"\n=== seed {s} ===")
        model, _ = train(cfg, device=device, log_fn=log_fn)
        entry = {"seed": int(s)}
        entry["source"] = evaluate_dataset(model, cfg.source_root, cfg, "Test",
                                           device, verbose=False)
        if cfg.target_root:
            entry["target"] = evaluate_dataset(model, cfg.target_root, cfg, "Test",
                                               device, verbose=False)
        runs.append(entry)
        del model
        if device == "cuda":
            torch.cuda.empty_cache()

    agg = {}
    for domain in ("source", "target"):
        if not all(domain in r for r in runs):
            continue
        agg[domain] = {m: {"mean": float(np.mean([r[domain][m] for r in runs])),
                           "std": float(np.std([r[domain][m] for r in runs], ddof=1))
                           if len(runs) > 1 else 0.0}
                       for m in METRICS}
    return {"runs": runs, "aggregate": agg}


def run_scale_generalisation(cfg: Config, ckpt: str, scales: Sequence[int] = (4, 8),
                             device: str = "cuda", log_fn=print) -> List[Dict]:
    """Evaluate a single trained model at scale factors it was not trained on.

    The v1 benchmark compared methods that were each run at a different scale
    factor (4, 8, 16 and 32), which is not a comparison at all. Here the factor
    is an explicit, reported axis.
    """
    out = []
    for s in scales:
        model, mcfg, _ = load_checkpoint(ckpt, device)
        mcfg.scale = s
        # the learned upsampler is tied to its training factor; rebuilding at a
        # new factor is only valid for the fixed-degradation evaluation path
        if s != cfg.scale:
            log_fn(f"  note: model was trained at x{cfg.scale}; evaluating at x{s} "
                   f"measures degradation-generalisation, not retrained performance")
        try:
            m = evaluate_dataset(model, mcfg.source_root, mcfg, "Test", device,
                                 verbose=False)
            out.append({"scale": s, **m})
        except Exception as exc:                       # shape mismatch at other factors
            log_fn(f"  x{s} not evaluable for this checkpoint: {exc}")
        del model
        if device == "cuda":
            torch.cuda.empty_cache()
    return out


# ----------------------------------------------------------------------- tables
def markdown_table(headers: Sequence[str], rows: Sequence[Sequence]) -> str:
    head = "| " + " | ".join(str(h) for h in headers) + " |"
    sep = "|" + "|".join("---" for _ in headers) + "|"
    body = "\n".join("| " + " | ".join(str(c) for c in r) + " |" for r in rows)
    return "\n".join([head, sep, body])


def latex_table(headers: Sequence[str], rows: Sequence[Sequence],
                caption: str = "", label: str = "") -> str:
    cols = "l" + "c" * (len(headers) - 1)
    lines = [r"\begin{table}[t]", r"\centering",
             rf"\caption{{{caption}}}", rf"\label{{{label}}}",
             rf"\begin{{tabular}}{{{cols}}}", r"\toprule",
             " & ".join(str(h) for h in headers) + r" \\", r"\midrule"]
    lines += [" & ".join(str(c) for c in r) + r" \\" for r in rows]
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(lines)


def comparison_table(entries: Dict[str, Dict[str, float]], fmt: str = "markdown",
                     caption: str = "", label: str = "") -> str:
    """entries: {method name -> {psnr, ssim, sam, ergas}} rendered with the
    best value in each column marked."""
    headers = ["Method", "PSNR (dB) up", "SSIM up", "SAM (deg) down", "ERGAS down"]
    best = {m: (max if HIGHER_IS_BETTER[m] else min)(
        e[m] for e in entries.values() if m in e) for m in METRICS}
    rows = []
    for name, e in entries.items():
        cells = [name]
        for m, prec in zip(METRICS, (3, 4, 3, 3)):
            val = e.get(m)
            if val is None:
                cells.append("-")
                continue
            txt = f"{val:.{prec}f}"
            if abs(val - best[m]) < 1e-9:
                txt = f"**{txt}**" if fmt == "markdown" else rf"\textbf{{{txt}}}"
            cells.append(txt)
        rows.append(cells)
    return (markdown_table(headers, rows) if fmt == "markdown"
            else latex_table(headers, rows, caption, label))


def ablation_table(results: List[Dict], fmt: str = "markdown") -> str:
    has_target = any("target" in r for r in results)
    headers = ["Variant", "Params (M)", "PSNR", "SAM", "ERGAS"]
    if has_target:
        headers += ["PSNR (cross)", "SAM (cross)", "ERGAS (cross)"]
    rows = []
    for r in results:
        cells = [r["variant"], f"{r['params_M']:.2f}",
                 f"{r['source']['psnr']:.3f}", f"{r['source']['sam']:.3f}",
                 f"{r['source']['ergas']:.3f}"]
        if has_target:
            t = r.get("target", {})
            cells += [f"{t.get('psnr', float('nan')):.3f}",
                      f"{t.get('sam', float('nan')):.3f}",
                      f"{t.get('ergas', float('nan')):.3f}"]
        rows.append(cells)
    return (markdown_table(headers, rows) if fmt == "markdown"
            else latex_table(headers, rows, "Component ablation.", "tab:ablation"))


def significance_table(comparisons: List[Dict], metric: str = "psnr",
                       fmt: str = "markdown") -> str:
    headers = ["Baseline", f"ours {metric}", f"baseline {metric}", "delta",
               "Wilcoxon p", "Cohen d", "n"]
    rows = []
    for c in comparisons:
        e = c[metric]
        star = "***" if e["p"] < 0.001 else "**" if e["p"] < 0.01 else \
               "*" if e["p"] < 0.05 else "n.s."
        rows.append([c["name_b"], f"{e['mean_a']:.3f}", f"{e['mean_b']:.3f}",
                     f"{e['delta']:+.3f}", f"{e['p']:.4f} {star}",
                     f"{e['d']:.2f}", int(c["n_scenes"])])
    return (markdown_table(headers, rows) if fmt == "markdown"
            else latex_table(headers, rows, f"Paired significance on {metric}.",
                             f"tab:sig_{metric}"))


# --------------------------------------------------------------------- reports
def environment_report() -> Dict[str, str]:
    """Everything a reviewer needs to reproduce the numbers."""
    import platform
    import sys
    info = {"python": sys.version.split()[0], "platform": platform.platform(),
            "torch": torch.__version__, "numpy": np.__version__,
            "cuda_available": str(torch.cuda.is_available())}
    if torch.cuda.is_available():
        info["gpu"] = torch.cuda.get_device_name(0)
        info["cuda"] = torch.version.cuda or "unknown"
        info["gpu_mem_GB"] = f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f}"
    return info


def save_results(path: str, payload: Dict) -> str:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

    def default(o):
        if isinstance(o, (np.floating, np.integer)):
            return o.item()
        if isinstance(o, np.ndarray):
            return o.tolist()
        return str(o)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=1, default=default)
    return path


def write_report(path: str, title: str, sections: List[Tuple[str, str]]) -> str:
    """Assemble a Markdown report from (heading, body) pairs."""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    parts = [f"# {title}", ""]
    for heading, body in sections:
        parts += [f"## {heading}", "", body, ""]
    text = "\n".join(parts)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    return text

In [ ]:
%%writefile daetf/selfcheck.py
"""Numerical self-checks.

Each of these turns a claim from the design document into something a reviewer
can run. They execute in a few seconds on CPU and are the first cell of the
Kaggle notebook, so a broken environment fails loudly and immediately rather
than 3 hours into training.
"""

from __future__ import annotations

import torch

from .config import Config
from .degrade import blur_downsample, gaussian_kernel2d
from .engine import test_time_adapt, tiled_inference
from .losses import SPCLoss
from .metrics import evaluate_arrays
from .model import DAETFNet
from .modules import (EquivariantFeatureExtractor, HaarDWT,
                      TensorSpectralSpatialEncoder)


def check_equivariance(device: str = "cpu", tol: float = 1e-4) -> float:
    """rot90(EFE(x)) must equal EFE(rot90(x))."""
    torch.manual_seed(0)
    efe = EquivariantFeatureExtractor(5, 8, 12, depth=2).to(device).eval()
    x = torch.randn(2, 5, 32, 32, device=device)
    with torch.no_grad():
        a = torch.rot90(efe(x), 1, (-2, -1))
        b = efe(torch.rot90(x, 1, (-2, -1)))
    err = float((a - b).abs().max())
    print(f"[check] p4 equivariance max|err| = {err:.3e} "
          f"({'PASS' if err < tol else 'FAIL'})")
    return err


def check_wavelet(tol: float = 1e-5) -> float:
    """IDWT(DWT(x)) must reconstruct x exactly (orthonormal Haar)."""
    dwt = HaarDWT()
    x = torch.randn(2, 7, 16, 16)
    err = float((dwt.inverse(dwt(x)) - x).abs().max())
    print(f"[check] Haar DWT reconstruction max|err| = {err:.3e} "
          f"({'PASS' if err < tol else 'FAIL'})")
    return err


def check_core_used() -> bool:
    """The Tucker core must receive gradient - the v1 bug was that it did not."""
    t = TensorSpectralSpatialEncoder(8, 8, 8, rank=4)
    a, b = torch.randn(1, 8, 8, 8), torch.randn(1, 8, 8, 8)
    t(a, b).sum().backward()
    ok = t.core.grad is not None and float(t.core.grad.abs().sum()) > 0
    print(f"[check] Tucker core receives gradient ({'PASS' if ok else 'FAIL'})")
    return ok


def check_observation_model(tol: float = 1e-5) -> bool:
    """Down(HR) must reproduce the LR observation the dataset built, otherwise
    the spatial-consistency term is penalising the wrong thing."""
    gt = torch.rand(1, 5, 64, 64)
    k = gaussian_kernel2d(9, 1.2, 1.2, 0.0)
    lr_a = blur_downsample(gt, k, 4)
    lr_b = blur_downsample(gt, k.unsqueeze(0), 4)     # per-sample kernel path
    err = float((lr_a - lr_b).abs().max())
    ok = err < tol and lr_a.shape[-1] == 16
    print(f"[check] observation model consistent, shape {tuple(lr_a.shape)}, "
          f"max|err| {err:.2e} ({'PASS' if ok else 'FAIL'})")
    return ok


def check_tta_isolation(device: str = "cpu", tol: float = 1e-6) -> bool:
    """Test-time adaptation must not leak between scenes.

    With restore=True the weights must come back exactly, so adapting scene B
    cannot inherit scene A's adaptation. Otherwise the reported cross-domain
    numbers would depend on the order the test scenes happen to be listed in.
    """
    torch.manual_seed(0)
    cfg = Config(patch=32, width=16, equi_width=4, rank=4, bands=31, msi_bands=3)
    model = DAETFNet(cfg).to(device)
    crit = SPCLoss(cfg, torch.rand(cfg.bands, cfg.msi_bands)).to(device)
    gt = torch.rand(1, cfg.bands, 32, 32, device=device)
    lr = blur_downsample(gt, gaussian_kernel2d(9, 1.2, 1.2, 0.0), cfg.scale)
    msi = torch.rand(1, cfg.msi_bands, 32, 32, device=device)

    before = {k: v.detach().clone() for k, v in model.state_dict().items()}
    first = test_time_adapt(model, lr, msi, crit, steps=3, restore=True)
    drift = max((v - before[k]).abs().max().item()
                for k, v in model.state_dict().items() if v.is_floating_point())
    second = test_time_adapt(model, lr, msi, crit, steps=3, restore=True)
    repeat = (first - second).abs().max().item()

    ok = drift < tol and repeat < tol
    print(f"[check] TTA isolation: weight drift {drift:.2e}, "
          f"repeat difference {repeat:.2e} ({'PASS' if ok else 'FAIL'})")
    return ok


def smoke_test(device: str = "cpu", check_ablations: bool = True) -> None:
    """End-to-end shape/gradient check on synthetic tensors."""
    cfg = Config(patch=32, width=32, equi_width=8, rank=8, batch=2,
                 bands=31, msi_bands=3)
    model = DAETFNet(cfg).to(device)
    srf = torch.rand(cfg.bands, cfg.msi_bands)
    crit = SPCLoss(cfg, srf).to(device)
    gt = torch.rand(2, cfg.bands, cfg.patch, cfg.patch, device=device)
    lr = blur_downsample(gt, gaussian_kernel2d(9, 1.2, 1.2, 0.0), cfg.scale)
    msi = torch.rand(2, cfg.msi_bands, cfg.patch, cfg.patch, device=device)

    out = model(lr, msi)
    assert out["out"].shape == gt.shape, (out["out"].shape, gt.shape)
    loss, logs = crit(out, gt, lr, msi, model, deg_gt=torch.rand(2, 5, device=device))
    loss.backward()
    grads = sum(1 for p in model.parameters() if p.grad is not None and p.grad.abs().sum() > 0)
    total = sum(1 for _ in model.parameters())
    print(f"[check] forward {tuple(out['out'].shape)}  loss {loss.item():.4f}  "
          f"params {model.n_params() / 1e6:.2f}M  tensors with grad {grads}/{total}")
    print(f"[check] loss terms: { {k: round(v, 4) for k, v in logs.items()} }")

    big_gt = torch.rand(1, cfg.bands, 96, 96, device=device)
    big_lr = blur_downsample(big_gt, gaussian_kernel2d(9, 1.2, 1.2, 0.0), cfg.scale)
    big_msi = torch.rand(1, cfg.msi_bands, 96, 96, device=device)
    pred = tiled_inference(model, big_lr, big_msi, cfg.scale, tile_hr=32, overlap=8)
    assert pred.shape == big_gt.shape, (pred.shape, big_gt.shape)
    m = evaluate_arrays(pred[0].detach().cpu().numpy().transpose(1, 2, 0),
                        big_gt[0].cpu().numpy().transpose(1, 2, 0), cfg.scale)
    print(f"[check] tiled inference {tuple(pred.shape)} PASS; "
          f"metrics { {k: round(v, 3) for k, v in m.items()} }")

    if check_ablations:
        for switch in ("use_equivariant", "use_tsse", "use_moe", "use_fdrm",
                       "use_backprojection", "use_degradation_code"):
            acfg = Config(patch=32, width=32, equi_width=8, rank=8, batch=2,
                          bands=31, msi_bands=3, **{switch: False})
            am = DAETFNet(acfg).to(device)
            ao = am(lr, msi)
            assert ao["out"].shape == gt.shape
            acrit = SPCLoss(acfg, srf).to(device)
            aloss, _ = acrit(ao, gt, lr, msi, am,
                             deg_gt=torch.rand(2, 5, device=device))
            aloss.backward()
            print(f"[check] ablation {switch}=False OK "
                  f"({am.n_params() / 1e6:.2f}M params)")


def run_all(device: str = "cpu") -> bool:
    """Every check. Returns True when all of them pass."""
    ok = True
    ok &= check_equivariance(device) < 1e-4
    ok &= check_wavelet() < 1e-5
    ok &= check_core_used()
    ok &= check_observation_model()
    ok &= check_tta_isolation(device)
    smoke_test(device)
    print(f"\n[selfcheck] {'ALL PASS' if ok else 'FAILURES PRESENT'}")
    return bool(ok)


if __name__ == "__main__":
    run_all()

In [ ]:
%%writefile daetf/__init__.py
"""DAETF-Net: Domain-Adaptive Equivariant Tensor Fusion Network.

Hyperspectral-multispectral image fusion built to survive the transfer from the
dataset it was trained on to one it has never seen.

    import daetf
    cfg = daetf.Config().resolve()          # finds the datasets, infers bands
    daetf.selfcheck.run_all()               # verifies the mechanisms numerically
    model, hist = daetf.train(cfg)
    daetf.evaluate_dataset(model, cfg.source_root, cfg)
"""

from . import experiments, selfcheck
from .config import Config
from .data import FusionPatchDataset, SceneCache, estimate_srf
from .degrade import FixedDegradation, blur_downsample, gaussian_kernel2d
from .engine import (cosine_lr, evaluate_dataset, evaluate_with_tta,
                     load_checkpoint, set_seed, test_time_adapt, tiled_inference,
                     train)
from .io_utils import (available_splits, discover_dataset, find_pairs,
                       infer_channels, load_mat, search_roots, to_chw01)
from .losses import (SPCLoss, charbonnier, gradient_loss, mmd_rbf, sam_loss)
from .metrics import (evaluate_arrays, metric_ergas, metric_psnr, metric_sam,
                      metric_ssim, ssim_torch)
from .model import DAETFNet
from .modules import (BackProjectionUpsampler, BicubicUpsampler,
                      DegradationEncoder, EquivariantFeatureExtractor, FiLM,
                      FrequencyDomainRefinement, HaarDWT, P4ConvP4, P4ConvZ2,
                      PlainFeatureExtractor, RegionAwareMoE,
                      TensorSpectralSpatialEncoder)

__version__ = "2.0.0"

__all__ = [
    "Config", "DAETFNet", "SPCLoss",
    "train", "evaluate_dataset", "evaluate_with_tta", "test_time_adapt",
    "tiled_inference", "load_checkpoint", "set_seed", "cosine_lr",
    "FusionPatchDataset", "SceneCache", "estimate_srf",
    "FixedDegradation", "blur_downsample", "gaussian_kernel2d",
    "discover_dataset", "find_pairs", "infer_channels", "available_splits",
    "load_mat", "to_chw01", "search_roots",
    "charbonnier", "sam_loss", "gradient_loss", "mmd_rbf",
    "evaluate_arrays", "metric_psnr", "metric_ssim", "metric_sam",
    "metric_ergas", "ssim_torch",
    "P4ConvZ2", "P4ConvP4", "EquivariantFeatureExtractor",
    "PlainFeatureExtractor", "DegradationEncoder", "FiLM",
    "TensorSpectralSpatialEncoder", "RegionAwareMoE", "HaarDWT",
    "FrequencyDomainRefinement", "BackProjectionUpsampler", "BicubicUpsampler",
    "experiments", "selfcheck", "__version__",
]

In [ ]:
import importlib, sys
for m in [k for k in list(sys.modules) if k.startswith('daetf')]:
    del sys.modules[m]
sys.path.insert(0, os.getcwd())
import daetf
print('daetf', daetf.__version__, 'loaded from', os.path.dirname(daetf.__file__))

## 3. Self-checks

Each claim in the architecture is verified numerically before any training
starts, so a broken environment fails in seconds rather than three hours in.

In [ ]:
ok = daetf.selfcheck.run_all(device='cpu')
assert ok, 'self-checks failed - stop here'

## 4. Configuration

Dataset roots and band counts are discovered from the filesystem - nothing is
hardcoded. Attach the CAVE and Harvard datasets and this cell finds them.
If discovery misses, set `DAETF_DATA_ROOTS` or pass `source_root=` explicitly.

In [ ]:
QUICK = False          # True -> a few minutes end-to-end, for validating the pipeline
DEVICE = 'cuda' if (torch.cuda.is_available() and GPU_OK) else 'cpu'
if DEVICE == 'cpu' and torch.cuda.is_available():
    raise RuntimeError(
        'The attached GPU is not supported by this torch build (see the '
        'environment cell). Switch the accelerator to "GPU T4 x2" and re-run. '
        'Set DEVICE = "cpu" manually only if you intend a CPU run.')

cfg = daetf.Config(
    scale=4,                # fixed for every method, unlike the v1 benchmark
    patch=64,
    batch=16,
    iters=20000,
    width=64,
    amp=True,               # fp16: halves memory, faster on T4
    workers=2,
    out_dir=os.path.join(os.getcwd(), 'daetf_out'),
    val_every=1000,
    log_every=100,
)
if QUICK:
    cfg.iters, cfg.batch, cfg.val_every, cfg.log_every = 300, 8, 150, 50

cfg.resolve()           # auto-discovery: finds CAVE as source, Harvard as target
print(json.dumps({k: v for k, v in cfg.to_dict().items()
                  if k in ('source_root','target_root','bands','msi_bands',
                           'scale','patch','batch','iters')}, indent=1))

In [ ]:
# what the loaders actually see
for name, root in (('source', cfg.source_root), ('target', cfg.target_root)):
    if not root:
        continue
    splits = daetf.available_splits(root)
    line = [f'{name}: {root}']
    for canon in ('Train', 'Test'):
        if canon in splits:
            line.append(f'{canon}={len(daetf.find_pairs(root, canon))} scenes')
    print('  '.join(line))

## 5. Training

Trained on the source domain only. When a target root is present, unlabelled
target patches are drawn alongside and aligned with an MMD penalty - **no target
ground truth is ever used**, so the cross-domain evaluation below stays honest.

In [ ]:
t0 = time.time()
model, history = daetf.train(cfg, device=DEVICE)
print(f'\ntrained in {(time.time() - t0) / 60:.1f} min')

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(history['iter'], history['loss'])
ax[0].set_xlabel('iteration'); ax[0].set_ylabel('SPC loss'); ax[0].grid(alpha=.3)
ax[0].set_title('training loss')
if history['val']:
    it = [v['iter'] for v in history['val']]
    ax[1].plot(it, [v['psnr'] for v in history['val']], marker='o', label='PSNR')
    ax[1].set_xlabel('iteration'); ax[1].set_ylabel('PSNR (dB)'); ax[1].grid(alpha=.3)
    axb = ax[1].twinx(); axb.plot(it, [v['sam'] for v in history['val']],
                                  marker='s', color='tab:red', label='SAM')
    axb.set_ylabel('SAM (deg)')
    ax[1].set_title('validation')
plt.tight_layout(); plt.savefig('training_curves.png', dpi=140); plt.show()

## 6. In-domain evaluation

Full scenes via Hann-weighted overlapping tiles. All four metrics come from one
shared implementation with a fixed `data_range=1.0` - never the per-image
maximum, which is what inflated several v1 numbers on dark scenes.

In [ ]:
print('=== in-domain (source Test) ===')
src_mean, src_rows = daetf.evaluate_dataset(
    model, cfg.source_root, cfg, 'Test', DEVICE, return_rows=True)

## 7. Cross-domain transfer - the actual research question

Zero-shot on the target dataset, then the same model adapted per scene using
**only** the two physics terms. No target ground truth is used at any point.

In [ ]:
tgt_mean = tgt_rows = tta_mean = tta_rows = None
if cfg.target_root:
    print('=== zero-shot cross-domain (target Test) ===')
    tgt_mean, tgt_rows = daetf.evaluate_dataset(
        model, cfg.target_root, cfg, 'Test', DEVICE, return_rows=True)

    print('\n=== cross-domain + self-supervised test-time adaptation ===')
    import numpy as np
    srf = np.load('daetf_srf.npy') if os.path.exists('daetf_srf.npy') else \
          torch.load(os.path.join(cfg.out_dir, 'daetf_final.pth'),
                     map_location='cpu', weights_only=False)['srf']
    tta_mean, tta_rows = daetf.evaluate_with_tta(
        model, cfg.target_root, cfg, srf, 'Test', DEVICE, steps=30)
else:
    print('no target dataset attached - skipping the transfer study')

In [ ]:
from daetf.experiments import comparison_table, summarise_rows
entries = {'DAETF-Net (in-domain)': src_mean}
if tgt_mean: entries['DAETF-Net (zero-shot transfer)'] = tgt_mean
if tta_mean: entries['DAETF-Net (transfer + TTA)'] = tta_mean
print(comparison_table(entries))

print('\nper-metric spread (mean, std, bootstrap 95% CI):')
for name, rows in (('in-domain', src_rows), ('transfer', tgt_rows),
                   ('transfer+TTA', tta_rows)):
    if not rows: continue
    s = summarise_rows(rows)
    print(f'  {name}:')
    for m in ('psnr', 'ssim', 'sam', 'ergas'):
        e = s[m]
        print(f'    {m:6s} {e["mean"]:9.4f} +/- {e["std"]:7.4f}   '
              f'CI [{e["ci_lo"]:.4f}, {e["ci_hi"]:.4f}]  n={e["n"]}')

## 8. Comparison against the published baselines

The numbers below were recorded by the ten baseline notebooks in `existing/`.

**Read this table with care.** Those runs each used their own protocol - scale
factors of 4, 8, 16 and 32, different normalisations, and ERGAS scale arguments
that did not always match the actual downsampling. They are reproduced here for
reference, but a like-for-like claim requires re-running each baseline under
this protocol; `existing/results/` documents the discrepancies. Rows marked
`same-protocol` are the only strictly comparable ones.

In [ ]:
# recorded by the baseline notebooks in existing/ (their own protocols)
BASELINES_SOURCE = {
    'Fusformer  (x4)':  {'psnr': 50.20, 'ssim': 0.9996, 'sam': 2.35, 'ergas': 0.85},
    'DHIF-Net   (x8)':  {'psnr': 48.80, 'ssim': 0.9966, 'sam': 2.16, 'ergas': 0.51},
    'DBIN       (x16)': {'psnr': 47.14, 'ssim': 0.9939, 'sam': 2.97, 'ergas': 0.33},
    'TSFN       (x8)':  {'psnr': 46.40, 'ssim': 0.9943, 'sam': 2.75, 'ergas': 0.63},
    'UTAL       (x32)': {'psnr': 41.37, 'ssim': 0.9906, 'sam': 4.62, 'ergas': 0.27},
    'MoG-DCN    (x32)': {'psnr': 38.55, 'ssim': 0.9715, 'sam': 6.62, 'ergas': 0.38},
    'PSRT       (x32)': {'psnr': 38.24, 'ssim': 0.9647, 'sam': 7.55, 'ergas': 0.39},
    'LRU        (x4)':  {'psnr': 37.16, 'ssim': 0.9768, 'sam': 3.73, 'ergas': 4.07},
    'AMGSGAN    (x4)':  {'psnr': 37.42, 'sam': 7.41, 'ergas': 2.00},
    'IFCASformer(CASSI)': {'psnr': 35.98, 'ssim': 0.9602, 'sam': 5.15, 'ergas': 3.55},
}
BASELINES_TARGET = {
    'DHIF-Net   (x8)':  {'psnr': 70.20, 'ssim': 0.9997, 'sam': 2.62, 'ergas': 0.93},
    'PSRT       (x32)': {'psnr': 57.25, 'ssim': 0.9918, 'sam': 15.87, 'ergas': 1.81},
    'TSFN       (x8)':  {'psnr': 56.35, 'ssim': 0.9935, 'sam': 8.29, 'ergas': 4.60},
    'UTAL       (x32)': {'psnr': 48.60, 'ssim': 0.9511, 'sam': 36.46, 'ergas': 16.60},
    'IFCASformer(CASSI)': {'psnr': 44.73, 'ssim': 0.8713, 'sam': 26.86, 'ergas': 85.41},
    'LRU        (x4)':  {'psnr': 38.76, 'ssim': 0.7320, 'sam': 7.77, 'ergas': 80.04},
    'AMGSGAN    (x4)':  {'psnr': 31.86, 'sam': 5.94, 'ergas': 7.16},
    'MoG-DCN    (x32)': {'psnr': 30.11, 'ssim': 0.9892, 'sam': 14.60, 'ergas': 4.64},
    'Fusformer  (x4)':  {'psnr': 25.80, 'ssim': 0.3059, 'sam': 58.89, 'ergas': 302.39},
}

print('=== SOURCE DOMAIN ===')
print(comparison_table({**BASELINES_SOURCE,
                        'DAETF-Net (ours, x%d, same-protocol)' % cfg.scale: src_mean}))
if tgt_mean:
    ours = {'DAETF-Net (ours, zero-shot)': tgt_mean}
    if tta_mean: ours['DAETF-Net (ours, +TTA)'] = tta_mean
    print('\n=== TARGET DOMAIN (cross-dataset transfer) ===')
    print(comparison_table({**BASELINES_TARGET, **ours}))

### Spectral degradation under transfer

The single number that matters for the research claim: how much worse each
method's *spectral* fidelity gets when the domain changes.

In [ ]:
rows = []
for name in set(BASELINES_SOURCE) & set(BASELINES_TARGET):
    s, t = BASELINES_SOURCE[name], BASELINES_TARGET[name]
    rows.append((name, s['sam'], t['sam'], t['sam'] - s['sam'],
                 s.get('ergas'), t.get('ergas')))
if tgt_mean:
    rows.append(('DAETF-Net (ours)', src_mean['sam'], tgt_mean['sam'],
                 tgt_mean['sam'] - src_mean['sam'],
                 src_mean['ergas'], tgt_mean['ergas']))
    if tta_mean:
        rows.append(('DAETF-Net (+TTA)', src_mean['sam'], tta_mean['sam'],
                     tta_mean['sam'] - src_mean['sam'],
                     src_mean['ergas'], tta_mean['ergas']))
rows.sort(key=lambda r: r[3])
print(f'{"method":<24}{"SAM src":>9}{"SAM tgt":>9}{"delta":>9}'
      f'{"ERGAS src":>11}{"ERGAS tgt":>11}')
print('-' * 73)
for n, ss, ts, d, es, et in rows:
    print(f'{n:<24}{ss:9.2f}{ts:9.2f}{d:+9.2f}'
          f'{(es if es is not None else float("nan")):11.2f}'
          f'{(et if et is not None else float("nan")):11.2f}')

## 9. Cost

Parameters, GFLOPs, latency and peak memory for one full scene. The earlier
benchmark reported none of these, so nothing could be judged per unit of compute.

In [ ]:
prof = daetf.experiments.profile_model(model, cfg, device=DEVICE, hr=512)
for k, v in prof.items():
    print(f'  {k:14s} {v:.3f}' if isinstance(v, float) else f'  {k:14s} {v}')

## 10. Component ablation

Each variant is retrained from scratch with an identical budget, seed and data
order, and each disabled module is replaced by a **matched control arm** (plain
convolutions for the equivariant stem, bicubic for back-projection, concat-fuse
for the Tucker interaction) rather than by nothing - otherwise the ablation
measures capacity, not the mechanism.

This retrains the model once per variant. Set `RUN_ABLATION = True` and expect
roughly 8x the single-run time.

In [ ]:
RUN_ABLATION = False
ablation = None
if RUN_ABLATION:
    abl_cfg = daetf.Config(**cfg.to_dict())
    abl_cfg.iters = max(2000, cfg.iters // 4)   # shorter budget, identical across variants
    ablation = daetf.experiments.run_ablation(abl_cfg, device=DEVICE)
    print('\n' + daetf.experiments.ablation_table(ablation))
else:
    print('ablation skipped (set RUN_ABLATION = True)')

## 11. What the model learned

The MoE gate is a per-pixel distribution over experts, so it can be shown as a
map: this is the interpretability the design document claimed and v1 could not
deliver, since its gate was a single global vector per image.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if getattr(model, 'moe', None) is not None:
    pairs = daetf.find_pairs(cfg.source_root, 'Test')
    cache = daetf.SceneCache(cfg.bands, cfg.msi_bands, limit=1)
    hsi, rgb = cache.get(*pairs[0])
    h = (hsi.shape[1] // cfg.scale) * cfg.scale
    w = (hsi.shape[2] // cfg.scale) * cfg.scale
    h, w = min(h, 256), min(w, 256)
    gt = torch.from_numpy(hsi[:, :h, :w].astype(np.float32))[None].to(DEVICE)
    msi = torch.from_numpy(rgb[:, :h, :w].astype(np.float32))[None].to(DEVICE)
    lr = daetf.FixedDegradation.from_config(cfg).to(DEVICE)(gt)
    model.eval()
    with torch.no_grad():
        out = model(lr, msi)
    gate = model.moe.last_gate[0].cpu().numpy()

    n = gate.shape[0]
    fig, axes = plt.subplots(1, n + 2, figsize=(3 * (n + 2), 3))
    axes[0].imshow(np.clip(msi[0].cpu().numpy().transpose(1, 2, 0), 0, 1))
    axes[0].set_title('MSI'); axes[0].axis('off')
    err = np.abs(out['out'][0].cpu().numpy() - gt[0].cpu().numpy()).mean(0)
    im = axes[1].imshow(err, cmap='inferno'); axes[1].set_title('abs error')
    axes[1].axis('off'); plt.colorbar(im, ax=axes[1], fraction=.046)
    for i in range(n):
        axes[i + 2].imshow(gate[i], cmap='viridis', vmin=0, vmax=1)
        axes[i + 2].set_title(f'expert {i}  ({gate[i].mean():.2f})')
        axes[i + 2].axis('off')
    plt.tight_layout(); plt.savefig('moe_routing.png', dpi=140); plt.show()
    print('expert usage:', np.round(gate.mean(axis=(1, 2)), 3),
          '(uniform would be', round(1 / n, 3), ')')

## 12. Save everything

Results, per-scene tables, the environment report and the checkpoints are
written to `/kaggle/working` so the run is reproducible and the numbers can be
pulled straight into the manuscript.

In [ ]:
from daetf.experiments import (environment_report, save_results, write_report,
                               comparison_table, summarise_rows)

payload = {
    'config': cfg.to_dict(),
    'environment': environment_report(),
    'params_M': model.n_params() / 1e6,
    'efficiency': prof,
    'source': {'mean': src_mean, 'rows': src_rows,
               'summary': summarise_rows(src_rows)},
}
if tgt_rows:
    payload['target_zeroshot'] = {'mean': tgt_mean, 'rows': tgt_rows,
                                  'summary': summarise_rows(tgt_rows)}
if tta_rows:
    payload['target_tta'] = {'mean': tta_mean, 'rows': tta_rows,
                             'summary': summarise_rows(tta_rows)}
if ablation:
    payload['ablation'] = [{k: v for k, v in r.items()
                            if not k.endswith('_rows')} for r in ablation]

save_results('results.json', payload)

sections = [('Environment', '\n'.join(f'- **{k}**: {v}'
             for k, v in environment_report().items()))]
entries = {'DAETF-Net (in-domain)': src_mean}
if tgt_mean: entries['DAETF-Net (zero-shot)'] = tgt_mean
if tta_mean: entries['DAETF-Net (+TTA)'] = tta_mean
sections.append(('Results', comparison_table(entries)))
sections.append(('Cost', '\n'.join(f'- **{k}**: {v}' for k, v in prof.items())))
if ablation:
    sections.append(('Ablation', daetf.experiments.ablation_table(ablation)))
write_report('RESULTS.md', 'DAETF-Net run report', sections)

print('written:', [f for f in os.listdir('.')
                   if f.endswith(('.json', '.md', '.png', '.pth'))])
print('checkpoints:', os.listdir(cfg.out_dir))